In [2]:
pip install pykeen 

  Obtaining dependency information for pykeen from https://files.pythonhosted.org/packages/10/53/7f38ff5559822f774b733ee20f908b7a227434e553b02d8e28c50afeda69/pykeen-1.11.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 4.1 MB/s eta 0:00:00
  Obtaining dependency information for click_default_group from https://files.pythonhosted.org/packages/2c/1a/aff8bb287a4b1400f69e09a53bd65de96aa5cee5691925b38731c67fc695/click_default_group-1.2.4-py2.py3-none-any.whl.metadata
  Obtaining dependency information for more_click from https://files.pythonhosted.org/packages/ad/8e/4e25da8883c5d661eaf4225a951046b8f4466e75eadb8594e204550b3179/more_click-0.1.2-py3-none-any.whl.metadata
  Obtaining dependency information for pystow>=0.4.3 from https://files.pythonhosted.org/packages/bb/9a/45876f864c12dd63841e6da2836117a1c5dc67ec8384feba188a7e55184d/pystow-0.7.0-py3-none-any.whl.metadata
  Obtaining dependency information for docdata>=0.0.5 from https://files.pythonhosted.

In [1]:
import numpy as np
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
import torch
import os


/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
import rdflib
from pykeen.triples import TriplesFactory

# Load RDF file
g = rdflib.Graph()
g.parse("mappings/NKB_RDF_V3.ttl", format="turtle")  # Can also use other formats like "xml", "n3", etc.
print('Loaded RDF file')
triples = []
for s, p, o in g:
    if isinstance(o, rdflib.URIRef):
        head = str(s)
        relation = str(p)
        tail = str(o)
        triples.append((head, relation, tail))
triples_array = np.array(triples, dtype=str)
# Create a PyKEEN TriplesFactory
tf = TriplesFactory.from_labeled_triples(triples_array)

Loaded RDF file


In [5]:
torch.save(tf, "nkb_triples_factory.pt")

In [15]:
# =============================================================================
# Triple Factory Validation and Entity Mapping Functions
# =============================================================================

def validate_triple_factory_entities(triples_factory_path="nkb_triples_factory.pt", 
                                    ttl_file="mappings/NKB_RDF_V3.ttl"):
    """
    Validate that the triple factory contains the entities we expect
    and create mappings for visualization
    """
    print("🔍 Validating Triple Factory against Original TTL...")
    
    try:
        # Load the triple factory with proper security settings for newer PyTorch
        import torch
        
        # Try different loading methods for compatibility
        try:
            # Method 1: Try with weights_only=False (if you trust the source)
            tf = torch.load(triples_factory_path, weights_only=False)
        except Exception as e1:
            try:
                # Method 2: Add safe globals for PyKEEN classes
                from pykeen.triples.triples_factory import TriplesFactory
                torch.serialization.add_safe_globals([TriplesFactory])
                tf = torch.load(triples_factory_path)
            except Exception as e2:
                try:
                    # Method 3: Use safe_globals context manager
                    from pykeen.triples.triples_factory import TriplesFactory
                    with torch.serialization.safe_globals([TriplesFactory]):
                        tf = torch.load(triples_factory_path)
                except Exception as e3:
                    # Method 4: Legacy loading for older PyTorch
                    tf = torch.load(triples_factory_path, map_location='cpu')
        
        print(f"✓ Loaded triple factory: {tf}")
        print(f"  - Entities: {tf.num_entities:,}")
        print(f"  - Relations: {tf.num_relations:,}")
        print(f"  - Triples: {tf.num_triples:,}")
        
        # Get entity and relation mappings from triple factory
        tf_entity_to_id = tf.entity_to_id
        tf_relation_to_id = tf.relation_to_id
        tf_id_to_entity = {v: k for k, v in tf_entity_to_id.items()}
        tf_id_to_relation = {v: k for k, v in tf_relation_to_id.items()}
        
        print(f"\n📊 Sample entities from triple factory:")
        for i, (entity, entity_id) in enumerate(list(tf_entity_to_id.items())[:10]):
            short_entity = entity.split('/')[-1].split('#')[-1][:50]
            print(f"  {i+1}. {short_entity} (ID: {entity_id})")
        
        print(f"\n📊 Sample relations from triple factory:")
        for i, (relation, relation_id) in enumerate(list(tf_relation_to_id.items())[:10]):
            short_relation = relation.split('/')[-1].split('#')[-1][:50]
            print(f"  {i+1}. {short_relation} (ID: {relation_id})")
        
        # Now let's check against the TTL file entities
        if ttl_file:
            try:
                print(f"\n🔍 Comparing with original TTL file: {ttl_file}")
                g = rdflib.Graph()
                g.parse(ttl_file, format="turtle")
                
                # Get all unique entities from TTL
                ttl_entities = set()
                ttl_relations = set()
                
                for s, p, o in g:
                    ttl_entities.add(str(s))
                    ttl_relations.add(str(p))
                    if isinstance(o, rdflib.URIRef):
                        ttl_entities.add(str(o))
                
                print(f"  - TTL Entities: {len(ttl_entities):,}")
                print(f"  - TTL Relations: {len(ttl_relations):,}")
                
                # Check coverage
                tf_entities = set(tf_entity_to_id.keys())
                tf_relations = set(tf_relation_to_id.keys())
                
                entity_coverage = len(tf_entities.intersection(ttl_entities)) / len(ttl_entities) * 100
                relation_coverage = len(tf_relations.intersection(ttl_relations)) / len(ttl_relations) * 100
                
                print(f"\n📈 Coverage Analysis:")
                print(f"  - Entity coverage: {entity_coverage:.1f}% ({len(tf_entities.intersection(ttl_entities)):,}/{len(ttl_entities):,})")
                print(f"  - Relation coverage: {relation_coverage:.1f}% ({len(tf_relations.intersection(ttl_relations)):,}/{len(ttl_relations):,})")
                
                # Find missing entities
                missing_entities = ttl_entities - tf_entities
                missing_relations = ttl_relations - tf_relations
                
                if missing_entities:
                    print(f"\n❌ Missing entities (showing first 10 of {len(missing_entities):,}):")
                    for i, entity in enumerate(list(missing_entities)[:10]):
                        short_entity = entity.split('/')[-1].split('#')[-1][:50]
                        print(f"  {i+1}. {short_entity}")
                
                if missing_relations:
                    print(f"\n❌ Missing relations (showing first 10 of {len(missing_relations):,}):")
                    for i, relation in enumerate(list(missing_relations)[:10]):
                        short_relation = relation.split('/')[-1].split('#')[-1][:50]
                        print(f"  {i+1}. {short_relation}")
                
                return {
                    'triple_factory': tf,
                    'tf_entity_to_id': tf_entity_to_id,
                    'tf_relation_to_id': tf_relation_to_id,
                    'tf_id_to_entity': tf_id_to_entity,
                    'tf_id_to_relation': tf_id_to_relation,
                    'ttl_entities': ttl_entities,
                    'ttl_relations': ttl_relations,
                    'entity_coverage': entity_coverage,
                    'relation_coverage': relation_coverage,
                    'missing_entities': missing_entities,
                    'missing_relations': missing_relations
                }
                
            except Exception as e:
                print(f"❌ Error analyzing TTL file: {e}")
                return {
                    'triple_factory': tf,
                    'tf_entity_to_id': tf_entity_to_id,
                    'tf_relation_to_id': tf_relation_to_id,
                    'tf_id_to_entity': tf_id_to_entity,
                    'tf_id_to_relation': tf_id_to_relation
                }
        else:
            return {
                'triple_factory': tf,
                'tf_entity_to_id': tf_entity_to_id,
                'tf_relation_to_id': tf_relation_to_id,
                'tf_id_to_entity': tf_id_to_entity,
                'tf_id_to_relation': tf_id_to_relation
            }
            
    except Exception as e:
        print(f"❌ Error loading triple factory: {e}")
        return None


In [16]:
def create_entity_type_mapping_from_nkb(tf_id_to_entity, ttl_file="mappings/NKB_RDF_V3.ttl"):
    """
    Create entity type mappings specifically for NKB entities based on the 
    entity types we extracted from the previous analysis
    """
    print("🏷️ Creating NKB entity type mappings...")
    
    # NKB-specific entity type patterns based on your previous analysis
    nkb_entity_patterns = {
        # Main entity types from your analysis
        'Material': ['NPO_199', 'material', 'nanoparticle', 'nanomaterial'],
        'Assay': ['BAO_0000179', 'assay', 'test', 'measurement'],
        'Result': ['NPO_1680', 'result', 'outcome', 'measurement'],
        'Publication': ['OBI_0000070', 'publication', 'paper', 'article', 'doi'],
        'Medium': ['medium', 'solution', 'buffer'],
        'Parameter': ['parameter', 'condition', 'setting'],
        'MaterialFG': ['materialfg', 'functional_group'],
        'Additive': ['additive', 'supplement'],
        'Contamination': ['contamination', 'contaminant'],
        'MolecularResult': ['molecular', 'molecule', 'compound'],
        
        # Ontology-based patterns
        'Concentration': ['concentration', 'C25372', 'amount'],
        'Unit': ['unit', 'UO_', 'measurement_unit'],
        'Chemical': ['CHEBI_', 'chemical', 'substance'],
        'Biological': ['biological', 'bio', 'organism'],
        'Physical': ['physical', 'property', 'characteristic'],
        'Temporal': ['time', 'duration', 'temporal'],
        'Spatial': ['location', 'position', 'spatial'],
        
        # Identifier patterns
        'Identifier': ['identifier', 'id', 'uuid', 'n7656432b49624fcfa673c580ed678c01b'],
        'Unknown': []  # Default category
    }
    
    # Color scheme for NKB entities
    nkb_colors = {
        'Material': '#FF6B6B',           # Red - core materials
        'Assay': '#4ECDC4',             # Teal - testing procedures  
        'Result': '#45B7D1',            # Blue - outcomes
        'Publication': '#96CEB4',        # Green - literature
        'Medium': '#FFEAA7',            # Yellow - experimental conditions
        'Parameter': '#DDA0DD',         # Plum - settings
        'MaterialFG': '#FFA07A',        # Light salmon - functional groups
        'Additive': '#98D8C8',          # Mint - supplements
        'Contamination': '#F7DC6F',     # Light yellow - contaminants
        'MolecularResult': '#BB8FCE',   # Light purple - molecular data
        'Concentration': '#85C1E9',     # Light blue - concentrations
        'Unit': '#F8C471',             # Light orange - units
        'Chemical': '#82E0AA',          # Light green - chemicals
        'Biological': '#F1948A',        # Light red - biological
        'Physical': '#D7BDE2',          # Light purple - physical
        'Temporal': '#AED6F1',          # Light blue - time
        'Spatial': '#F9E79F',           # Light yellow - space
        'Identifier': '#D5DBDB',        # Light gray - IDs
        'Unknown': '#CCCCCC'            # Gray - unknown
    }
    
    entity_types = {}
    type_counts = Counter()
    entity_colors = []
    entity_labels = []
    
    print(f"Analyzing {len(tf_id_to_entity)} entities...")
    
    for entity_id in range(len(tf_id_to_entity)):
        entity_uri = tf_id_to_entity[entity_id]
        entity_lower = entity_uri.lower()
        
        # Determine entity type based on patterns
        assigned_type = 'Unknown'
        
        # Check each pattern category
        for type_name, patterns in nkb_entity_patterns.items():
            if type_name == 'Unknown':
                continue
                
            for pattern in patterns:
                if pattern.lower() in entity_lower:
                    assigned_type = type_name
                    break
            
            if assigned_type != 'Unknown':
                break
        
        # Special handling for numbered entities (likely auto-generated IDs)
        if assigned_type == 'Unknown':
            if entity_uri.isdigit():
                assigned_type = 'Identifier'
            elif 'n7656432b49624fcfa673c580ed678c01b' in entity_uri:
                assigned_type = 'Identifier'
        
        entity_types[entity_uri] = {assigned_type}
        type_counts[assigned_type] += 1
        entity_colors.append(nkb_colors[assigned_type])
        entity_labels.append(assigned_type)
    
    print(f"\n📊 NKB Entity Type Distribution:")
    for entity_type, count in type_counts.most_common():
        percentage = (count / len(tf_id_to_entity)) * 100
        print(f"  {entity_type}: {count:,} entities ({percentage:.1f}%)")
    
    # Also try to get more detailed types from TTL if available
    if ttl_file:
        try:
            print(f"\n🔍 Enhancing with TTL type information...")
            g = rdflib.Graph()
            g.parse(ttl_file, format="turtle")
            
            ttl_type_counts = Counter()
            enhanced_count = 0
            
            for s, p, o in g:
                if str(p) == "http://www.w3.org/1999/02/22-rdf-syntax-ns#type":
                    entity = str(s)
                    entity_type = str(o)
                    
                    if entity in entity_types:
                        # Map TTL types to our categories
                        ttl_type_name = entity_type.split('/')[-1].split('#')[-1]
                        ttl_type_counts[ttl_type_name] += 1
                        
                        # Update if we have a more specific type
                        if ttl_type_name in ['NPO_199', 'NPO_1680', 'BAO_0000179', 'OBI_0000070']:
                            old_type = list(entity_types[entity])[0]
                            if old_type == 'Unknown' or old_type == 'Identifier':
                                new_type = {
                                    'NPO_199': 'Material',
                                    'NPO_1680': 'Result', 
                                    'BAO_0000179': 'Assay',
                                    'OBI_0000070': 'Publication'
                                }.get(ttl_type_name, old_type)
                                
                                if new_type != old_type:
                                    entity_types[entity] = {new_type}
                                    enhanced_count += 1
            
            print(f"  Enhanced {enhanced_count} entities with TTL type information")
            print(f"  TTL types found: {dict(ttl_type_counts.most_common(10))}")
            
        except Exception as e:
            print(f"❌ Error enhancing with TTL: {e}")
    
    # Recreate colors and labels after enhancement
    entity_colors = []
    entity_labels = []
    final_type_counts = Counter()
    
    for entity_id in range(len(tf_id_to_entity)):
        entity_uri = tf_id_to_entity[entity_id]
        entity_type = list(entity_types[entity_uri])[0]
        entity_colors.append(nkb_colors[entity_type])
        entity_labels.append(entity_type)
        final_type_counts[entity_type] += 1
    
    print(f"\n📊 Final NKB Entity Type Distribution:")
    for entity_type, count in final_type_counts.most_common():
        percentage = (count / len(tf_id_to_entity)) * 100
        print(f"  {entity_type}: {count:,} entities ({percentage:.1f}%)")
    
    return {
        'entity_types': entity_types,
        'type_counts': final_type_counts,
        'entity_colors': entity_colors,
        'entity_labels': entity_labels,
        'color_scheme': nkb_colors
    }


In [17]:
def load_nkb_embeddings_with_validation(entity_embeddings_path="results_nkb/complex_nkb_entity_embeddings.pt",
                                       relation_embeddings_path="results_nkb/complex_nkb_relation_embeddings.pt",
                                       entity_to_id_path="results_nkb/complex_nkb_entity_to_id.pt",
                                       relation_to_id_path="results_nkb/complex_nkb_relation_to_id.pt",
                                       triples_factory_path="nkb_triples_factory.pt",
                                       ttl_file="mappings/NKB_RDF_V3.ttl"):
    """
    Load NKB embeddings and validate they match the triple factory
    """
    print("🚀 Loading NKB embeddings with validation...")
    
    # First validate the triple factory
    validation_results = validate_triple_factory_entities(triples_factory_path, ttl_file)
    if not validation_results:
        return None
    
    # Load embeddings with proper security settings
    try:
        # Safe loading function for PyTorch files
        def safe_torch_load(path):
            try:
                return torch.load(path, weights_only=False)
            except:
                try:
                    return torch.load(path, map_location='cpu')
                except:
                    return torch.load(path)
        
        entity_embeddings = safe_torch_load(entity_embeddings_path).detach().cpu().numpy()
        relation_embeddings = safe_torch_load(relation_embeddings_path).detach().cpu().numpy()
        entity_to_id = safe_torch_load(entity_to_id_path)
        relation_to_id = safe_torch_load(relation_to_id_path)
        
        print(f"✓ Loaded embeddings:")
        print(f"  Entity embeddings: {entity_embeddings.shape}")
        print(f"  Relation embeddings: {relation_embeddings.shape}")
        
        # Create reverse mappings
        id_to_entity = {v: k for k, v in entity_to_id.items()}
        id_to_relation = {v: k for k, v in relation_to_id.items()}
        
        # Validate embeddings match triple factory
        tf_entity_to_id = validation_results['tf_entity_to_id']
        tf_relation_to_id = validation_results['tf_relation_to_id']
        
        if len(entity_to_id) != len(tf_entity_to_id):
            print(f"⚠️ Warning: Entity count mismatch!")
            print(f"  Embeddings: {len(entity_to_id)}")
            print(f"  Triple Factory: {len(tf_entity_to_id)}")
        
        if len(relation_to_id) != len(tf_relation_to_id):
            print(f"⚠️ Warning: Relation count mismatch!")
            print(f"  Embeddings: {len(relation_to_id)}")
            print(f"  Triple Factory: {len(tf_relation_to_id)}")
        
        # Create entity type mappings
        print(f"\n🏷️ Creating entity type mappings...")
        type_mapping = create_entity_type_mapping_from_nkb(id_to_entity, ttl_file)
        
        return {
            'entity_embeddings': entity_embeddings,
            'relation_embeddings': relation_embeddings,
            'entity_to_id': entity_to_id,
            'relation_to_id': relation_to_id,
            'id_to_entity': id_to_entity,
            'id_to_relation': id_to_relation,
            'validation_results': validation_results,
            'type_mapping': type_mapping
        }
        
    except Exception as e:
        print(f"❌ Error loading embeddings: {e}")
        return None

def quick_nkb_visualization(sample_size=3000, show_plotly=True):
    """
    Quick visualization specifically for NKB embeddings
    """
    print("🎨 Creating NKB-specific visualization...")
    
    # Load NKB data with validation
    nkb_data = load_nkb_embeddings_with_validation()
    if not nkb_data:
        return None
    
    entity_embeddings = nkb_data['entity_embeddings']
    id_to_entity = nkb_data['id_to_entity']
    type_mapping = nkb_data['type_mapping']
    
    entity_colors = type_mapping['entity_colors']
    entity_labels = type_mapping['entity_labels']
    
    # Create enhanced NKB visualization
    embeddings_2d, df = create_nkb_tsne_visualization(
        entity_embeddings, id_to_entity, entity_colors, entity_labels,
        nkb_data=nkb_data, sample_size=sample_size, title_suffix=" - NKB Entities by Type", 
        show_plotly=show_plotly
    )
    
    # Print summary statistics
    print(f"\n📊 NKB Visualization Summary:")
    print(f"  Total entities: {len(entity_embeddings):,}")
    print(f"  Visualized entities: {len(embeddings_2d):,}")
    print(f"  Entity types: {len(set(entity_labels))}")
    
    type_counts = pd.Series(entity_labels).value_counts()
    print(f"  Most common types in visualization:")
    for entity_type, count in type_counts.head(5).items():
        print(f"    {entity_type}: {count:,}")
    
    return {
        'embeddings_2d': embeddings_2d,
        'df': df,
        'nkb_data': nkb_data,
        'type_summary': type_counts
    }


In [18]:
import rdflib

In [19]:
# Fix PyTorch loading issues for newer versions
import torch
print(f"PyTorch version: {torch.__version__}")

# Add PyKEEN classes to safe globals if needed
try:
    from pykeen.triples.triples_factory import TriplesFactory
    torch.serialization.add_safe_globals([TriplesFactory])
    print("✓ Added PyKEEN TriplesFactory to safe globals")
except Exception as e:
    print(f"Note: Could not add safe globals: {e}")

# Test and validate your NKB triple factory and embeddings
print("\n🔍 STEP 1: Validate Triple Factory")
print("="*50)
validation_results = validate_triple_factory_entities()


PyTorch version: 2.7.0
✓ Added PyKEEN TriplesFactory to safe globals

🔍 STEP 1: Validate Triple Factory
🔍 Validating Triple Factory against Original TTL...
✓ Loaded triple factory: TriplesFactory(num_entities=132844, num_relations=24, create_inverse_triples=False, num_triples=246972)
  - Entities: 132,844
  - Relations: 24
  - Triples: 246,972

📊 Sample entities from triple factory:
  1. data_0006 (ID: 0)
  2. data_1047 (ID: 1)
  3. data_1052 (ID: 2)
  4. data_1088 (ID: 3)
  5. data_1188 (ID: 4)
  6. data_2044 (ID: 5)
  7. data_2139 (ID: 6)
  8. data_2526 (ID: 7)
  9. data_2600 (ID: 8)
  10. data_2849 (ID: 9)

📊 Sample relations from triple factory:
  1. data_2909 (ID: 0)
  2. C25365 (ID: 1)
  3. C25372 (ID: 2)
  4. C25480 (ID: 3)
  5. C25704 (ID: 4)
  6. C40976 (ID: 5)
  7. C42614 (ID: 6)
  8. C43530 (ID: 7)
  9. C68553 (ID: 8)
  10. C70848 (ID: 9)

🔍 Comparing with original TTL file: mappings/NKB_RDF_V3.ttl
  - TTL Entities: 291,170
  - TTL Relations: 71

📈 Coverage Analysis:
  - Ent

In [22]:
from collections import Counter, defaultdict

In [23]:
# STEP 2: Load and validate embeddings with entity types
print("\n🚀 STEP 2: Load Embeddings with Validation")
print("="*50)
nkb_data = load_nkb_embeddings_with_validation(
       entity_embeddings_path="./results_nkb/complex_nkb_entity_embeddings.pt",
       relation_embeddings_path="./results_nkb/complex_nkb_relation_embeddings.pt",
       entity_to_id_path="./results_nkb/complex_nkb_entity_to_id.pt",
       relation_to_id_path="./results_nkb/complex_nkb_relation_to_id.pt"
   )



🚀 STEP 2: Load Embeddings with Validation
🚀 Loading NKB embeddings with validation...
🔍 Validating Triple Factory against Original TTL...
✓ Loaded triple factory: TriplesFactory(num_entities=132844, num_relations=24, create_inverse_triples=False, num_triples=246972)
  - Entities: 132,844
  - Relations: 24
  - Triples: 246,972

📊 Sample entities from triple factory:
  1. data_0006 (ID: 0)
  2. data_1047 (ID: 1)
  3. data_1052 (ID: 2)
  4. data_1088 (ID: 3)
  5. data_1188 (ID: 4)
  6. data_2044 (ID: 5)
  7. data_2139 (ID: 6)
  8. data_2526 (ID: 7)
  9. data_2600 (ID: 8)
  10. data_2849 (ID: 9)

📊 Sample relations from triple factory:
  1. data_2909 (ID: 0)
  2. C25365 (ID: 1)
  3. C25372 (ID: 2)
  4. C25480 (ID: 3)
  5. C25704 (ID: 4)
  6. C40976 (ID: 5)
  7. C42614 (ID: 6)
  8. C43530 (ID: 7)
  9. C68553 (ID: 8)
  10. C70848 (ID: 9)

🔍 Comparing with original TTL file: mappings/NKB_RDF_V3.ttl
  - TTL Entities: 291,170
  - TTL Relations: 71

📈 Coverage Analysis:
  - Entity coverage: 45.

In [24]:

# STEP 3: Create NKB-specific visualization
print("\n🎨 STEP 3: Create NKB Visualization")
print("="*50)
viz_results = quick_nkb_visualization(sample_size=2000, show_plotly=True)



🎨 STEP 3: Create NKB Visualization
🎨 Creating NKB-specific visualization...
🚀 Loading NKB embeddings with validation...
🔍 Validating Triple Factory against Original TTL...
✓ Loaded triple factory: TriplesFactory(num_entities=132844, num_relations=24, create_inverse_triples=False, num_triples=246972)
  - Entities: 132,844
  - Relations: 24
  - Triples: 246,972

📊 Sample entities from triple factory:
  1. data_0006 (ID: 0)
  2. data_1047 (ID: 1)
  3. data_1052 (ID: 2)
  4. data_1088 (ID: 3)
  5. data_1188 (ID: 4)
  6. data_2044 (ID: 5)
  7. data_2139 (ID: 6)
  8. data_2526 (ID: 7)
  9. data_2600 (ID: 8)
  10. data_2849 (ID: 9)

📊 Sample relations from triple factory:
  1. data_2909 (ID: 0)
  2. C25365 (ID: 1)
  3. C25372 (ID: 2)
  4. C25480 (ID: 3)
  5. C25704 (ID: 4)
  6. C40976 (ID: 5)
  7. C42614 (ID: 6)
  8. C43530 (ID: 7)
  9. C68553 (ID: 8)
  10. C70848 (ID: 9)

🔍 Comparing with original TTL file: mappings/NKB_RDF_V3.ttl
  - TTL Entities: 291,170
  - TTL Relations: 71

📈 Coverage 

NameError: name 'create_tsne_visualization' is not defined

In [29]:
# =============================================================================
# Helper Functions to Choose Between RGCN and Complex Embeddings
# =============================================================================

def list_available_embeddings():
    """
    List all available embedding files in your results directories
    """
    print("🔍 Available Embedding Files:")
    print("="*50)
    
    import os
    
    # Check results_nkb directory
    if os.path.exists("results_nkb"):
        print("\n📁 results_nkb/ directory:")
        files = os.listdir("results_nkb")
        embedding_files = [f for f in files if f.endswith('.pt') and ('embedding' in f or 'to_id' in f)]
        for f in sorted(embedding_files):
            size = os.path.getsize(f"results_nkb/{f}") / (1024*1024)  # MB
            print(f"  ✓ {f} ({size:.1f} MB)")
    
    # Check results_nkb_complex directory
    if os.path.exists("results_nkb_complex"):
        print("\n📁 results_nkb_complex/ directory:")
        files = os.listdir("results_nkb_complex")
        embedding_files = [f for f in files if f.endswith('.pt') and ('embedding' in f or 'to_id' in f)]
        for f in sorted(embedding_files):
            size = os.path.getsize(f"results_nkb_complex/{f}") / (1024*1024)  # MB
            print(f"  ✓ {f} ({size:.1f} MB)")
    
    print("\n💡 Available embedding types:")
    print("  • Complex embeddings: complex_nkb_*")
    print("  • RGCN embeddings: rgcn_nkb_*")

def load_complex_embeddings():
    """Load Complex embeddings specifically"""
    return load_nkb_embeddings_with_validation(
        entity_embeddings_path="results_nkb/complex_nkb_entity_embeddings.pt",
        relation_embeddings_path="results_nkb/complex_nkb_relation_embeddings.pt",
        entity_to_id_path="results_nkb/complex_nkb_entity_to_id.pt",
        relation_to_id_path="results_nkb/complex_nkb_relation_to_id.pt"
    )

def load_rgcn_embeddings():
    """Load RGCN embeddings specifically"""
    return load_nkb_embeddings_with_validation(
        entity_embeddings_path="results_nkb/rgcn_nkb_entity_embeddings.pt",
        relation_embeddings_path="results_nkb/rgcn_nkb_relation_embeddings.pt", 
        entity_to_id_path="results_nkb/rgcn_nkb_entity_to_id.pt",
        relation_to_id_path="results_nkb/rgcn_nkb_relation_to_id.pt"
    )

def compare_embedding_models(sample_size=1500):
    """Compare Complex vs RGCN embeddings side by side"""
    print("🔄 Loading and comparing both embedding models...")
    
    # Load both models
    print("\n📊 Loading Complex embeddings...")
    complex_data = load_complex_embeddings()
    
    print("\n📊 Loading RGCN embeddings...")  
    rgcn_data = load_rgcn_embeddings()
    
    if not complex_data or not rgcn_data:
        print("❌ Could not load both embedding types")
        return None
    
    # Compare dimensions
    print("\n📏 Embedding Dimensions:")
    print(f"  Complex: {complex_data['entity_embeddings'].shape}")
    print(f"  RGCN: {rgcn_data['entity_embeddings'].shape}")
    
    # Create side-by-side visualizations
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))
    
    # Sample entities for both
    n_entities = min(len(complex_data['id_to_entity']), len(rgcn_data['id_to_entity']))
    if n_entities > sample_size:
        indices = np.random.choice(n_entities, sample_size, replace=False)
    else:
        indices = np.arange(n_entities)
    
    # Complex embeddings visualization
    complex_embeddings_sampled = complex_data['entity_embeddings'][indices]
    complex_colors = [complex_data['type_mapping']['entity_colors'][i] for i in indices]
    complex_labels = [complex_data['type_mapping']['entity_labels'][i] for i in indices]
    
    tsne1 = TSNE(n_components=2, random_state=42, perplexity=min(30, len(indices)-1))
    complex_2d = tsne1.fit_transform(complex_embeddings_sampled)
    
    # Plot Complex
    unique_labels = list(set(complex_labels))
    for label in unique_labels:
        mask = np.array(complex_labels) == label
        if np.any(mask):
            color = complex_colors[np.where(mask)[0][0]]
            ax1.scatter(complex_2d[mask, 0], complex_2d[mask, 1], 
                       c=color, label=label, alpha=0.7, s=20)
    
    ax1.set_title('Complex Embeddings', fontsize=14)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # RGCN embeddings visualization  
    rgcn_embeddings_sampled = rgcn_data['entity_embeddings'][indices]
    rgcn_colors = [rgcn_data['type_mapping']['entity_colors'][i] for i in indices]
    rgcn_labels = [rgcn_data['type_mapping']['entity_labels'][i] for i in indices]
    
    tsne2 = TSNE(n_components=2, random_state=42, perplexity=min(30, len(indices)-1))
    rgcn_2d = tsne2.fit_transform(rgcn_embeddings_sampled)
    
    # Plot RGCN
    for label in unique_labels:
        mask = np.array(rgcn_labels) == label
        if np.any(mask):
            color = rgcn_colors[np.where(mask)[0][0]]
            ax2.scatter(rgcn_2d[mask, 0], rgcn_2d[mask, 1], 
                       c=color, label=label, alpha=0.7, s=20)
    
    ax2.set_title('RGCN Embeddings', fontsize=14)
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'complex_data': complex_data,
        'rgcn_data': rgcn_data,
        'complex_2d': complex_2d,
        'rgcn_2d': rgcn_2d
    }

# List available embeddings
list_available_embeddings()


🔍 Available Embedding Files:

📁 results_nkb/ directory:
  ✓ complex_nkb_entity_embeddings.pt (101.4 MB)
  ✓ complex_nkb_entity_to_id.pt (6.0 MB)
  ✓ complex_nkb_relation_embeddings.pt (0.0 MB)
  ✓ complex_nkb_relation_to_id.pt (0.0 MB)
  ✓ rgcn_nkb_entity_embeddings.pt (60.8 MB)
  ✓ rgcn_nkb_entity_to_id.pt (6.0 MB)
  ✓ rgcn_nkb_relation_embeddings.pt (0.0 MB)
  ✓ rgcn_nkb_relation_to_id.pt (0.0 MB)

📁 results_nkb_complex/ directory:
  ✓ complex_nkb_entity_embeddings.pt (101.4 MB)
  ✓ complex_nkb_entity_to_id.pt (6.0 MB)
  ✓ complex_nkb_relation_embeddings.pt (0.0 MB)
  ✓ complex_nkb_relation_to_id.pt (0.0 MB)

💡 Available embedding types:
  • Complex embeddings: complex_nkb_*
  • RGCN embeddings: rgcn_nkb_*


In [30]:
# =============================================================================
# UPDATED TEST CELLS WITH CORRECT PATHS
# =============================================================================

print("🚀 STEP 2: Load Embeddings with Validation (UPDATED)")
print("="*60)

# Choose which embeddings to load
print("Loading Complex embeddings...")
nkb_data = load_complex_embeddings()

# Alternative: Load RGCN embeddings instead
# nkb_data = load_rgcn_embeddings()

if nkb_data:
    print(f"\n✅ Successfully loaded NKB embeddings!")
    print(f"  Entity embeddings shape: {nkb_data['entity_embeddings'].shape}")
    print(f"  Relation embeddings shape: {nkb_data['relation_embeddings'].shape}")
    print(f"  Total entities: {len(nkb_data['id_to_entity']):,}")
    print(f"  Total relations: {len(nkb_data['id_to_relation']):,}")
    
    # Show entity type distribution
    type_counts = nkb_data['type_mapping']['type_counts']
    print(f"\n📊 Entity Type Distribution:")
    for entity_type, count in type_counts.most_common(10):
        percentage = (count / len(nkb_data['id_to_entity'])) * 100
        print(f"  {entity_type}: {count:,} entities ({percentage:.1f}%)")
else:
    print("❌ Failed to load embeddings")


🚀 STEP 2: Load Embeddings with Validation (UPDATED)
Loading Complex embeddings...
🚀 Loading NKB embeddings with validation...
🔍 Validating Triple Factory against Original TTL...
✓ Loaded triple factory: TriplesFactory(num_entities=132844, num_relations=24, create_inverse_triples=False, num_triples=246972)
  - Entities: 132,844
  - Relations: 24
  - Triples: 246,972

📊 Sample entities from triple factory:
  1. data_0006 (ID: 0)
  2. data_1047 (ID: 1)
  3. data_1052 (ID: 2)
  4. data_1088 (ID: 3)
  5. data_1188 (ID: 4)
  6. data_2044 (ID: 5)
  7. data_2139 (ID: 6)
  8. data_2526 (ID: 7)
  9. data_2600 (ID: 8)
  10. data_2849 (ID: 9)

📊 Sample relations from triple factory:
  1. data_2909 (ID: 0)
  2. C25365 (ID: 1)
  3. C25372 (ID: 2)
  4. C25480 (ID: 3)
  5. C25704 (ID: 4)
  6. C40976 (ID: 5)
  7. C42614 (ID: 6)
  8. C43530 (ID: 7)
  9. C68553 (ID: 8)
  10. C70848 (ID: 9)

🔍 Comparing with original TTL file: mappings/NKB_RDF_V3.ttl
  - TTL Entities: 291,170
  - TTL Relations: 71

📈 Cove

In [27]:
# =============================================================================
# Enhanced NKB-Specific Visualization Functions
# =============================================================================

def create_nkb_tsne_visualization(entity_embeddings, id_to_entity, entity_colors, entity_labels, 
                                 nkb_data=None, sample_size=5000, title_suffix="", show_plotly=True):
    """Create t-SNE visualization specifically optimized for NKB entities"""
    print(f"🎨 Creating NKB-specific t-SNE visualization...")
    
    # Import required libraries
    from sklearn.manifold import TSNE
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    from collections import Counter
    
    # Sample entities if too many, but ensure we get good representation of each type
    if len(entity_embeddings) > sample_size:
        # Stratified sampling to ensure each entity type is represented
        unique_labels = list(set(entity_labels))
        sampled_indices = []
        
        # Calculate samples per type
        samples_per_type = max(10, sample_size // len(unique_labels))
        remaining_samples = sample_size
        
        for label in unique_labels:
            label_indices = [i for i, l in enumerate(entity_labels) if l == label]
            if len(label_indices) > 0:
                n_samples = min(samples_per_type, len(label_indices), remaining_samples)
                if n_samples > 0:
                    selected = np.random.choice(label_indices, n_samples, replace=False)
                    sampled_indices.extend(selected)
                    remaining_samples -= n_samples
        
        # Fill remaining with random samples if needed
        if remaining_samples > 0:
            all_indices = set(range(len(entity_embeddings)))
            unused_indices = list(all_indices - set(sampled_indices))
            if unused_indices:
                additional = np.random.choice(unused_indices, 
                                            min(remaining_samples, len(unused_indices)), 
                                            replace=False)
                sampled_indices.extend(additional)
        
        indices = np.array(sampled_indices)
        sampled_embeddings = entity_embeddings[indices]
        colors = [entity_colors[i] for i in indices]
        labels = [entity_labels[i] for i in indices]
        entity_names = [id_to_entity[i] for i in indices]
    else:
        indices = np.arange(len(entity_embeddings))
        sampled_embeddings = entity_embeddings
        colors = entity_colors
        labels = entity_labels
        entity_names = [id_to_entity[i] for i in indices]
    
    # Apply t-SNE with optimized parameters for NKB data
    print("Applying t-SNE with NKB-optimized parameters...")
    perplexity = min(50, max(5, len(sampled_embeddings) // 4))
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, 
                n_iter=1000, learning_rate=200)
    embeddings_2d = tsne.fit_transform(sampled_embeddings)
    
    # Create enhanced entity information for NKB
    enhanced_entity_info = []
    for i, (entity_name, label) in enumerate(zip(entity_names, labels)):
        # Extract meaningful parts of NKB entity names
        short_name = entity_name.split('/')[-1].split('#')[-1]
        
        # Determine if it's an ID or a descriptive name
        entity_type_info = "ID" if short_name.isdigit() or 'n7656432b49624fcfa673c580ed678c01b' in short_name else "Named"
        
        # Try to extract meaningful identifiers
        if 'MaterialID' in entity_name:
            entity_category = "Material"
        elif 'AssayID' in entity_name:
            entity_category = "Assay"
        elif 'ResultID' in entity_name:
            entity_category = "Result"
        elif 'publication_DOI' in entity_name:
            entity_category = "Publication"
        elif any(ontology in entity_name for ontology in ['NPO_', 'BAO_', 'OBI_', 'CHEBI_']):
            entity_category = "Ontology Term"
        else:
            entity_category = label
        
        enhanced_entity_info.append({
            'short_name': short_name[:30],
            'full_name': entity_name,
            'type': label,
            'category': entity_category,
            'info_type': entity_type_info
        })
    
    # Create comprehensive DataFrame for NKB entities
    df = pd.DataFrame({
        'x': embeddings_2d[:, 0],
        'y': embeddings_2d[:, 1], 
        'color': colors,
        'label': labels,
        'short_name': [info['short_name'] for info in enhanced_entity_info],
        'full_name': [info['full_name'] for info in enhanced_entity_info],
        'category': [info['category'] for info in enhanced_entity_info],
        'info_type': [info['info_type'] for info in enhanced_entity_info]
    })
    
    # Enhanced Plotly visualization for NKB
    if show_plotly and plotly_available:
        try:
            
            # Create custom color scale for NKB entity types
            unique_labels = sorted(list(set(labels)))
            color_discrete_map = {}
            
            # Use the NKB color scheme if available
            if nkb_data and 'type_mapping' in nkb_data and 'color_scheme' in nkb_data['type_mapping']:
                nkb_colors = nkb_data['type_mapping']['color_scheme']
                for label in unique_labels:
                    color_discrete_map[label] = nkb_colors.get(label, '#CCCCCC')
            
            fig = px.scatter(df, x='x', y='y', color='label', 
                           hover_data=['short_name', 'category', 'info_type'],
                           title=f'NKB Knowledge Graph Entity Embeddings t-SNE{title_suffix}',
                           width=1200, height=800,
                           color_discrete_map=color_discrete_map if color_discrete_map else None)
            
            # Enhanced hover template for NKB entities
            fig.update_traces(
                marker=dict(size=6, opacity=0.8, line=dict(width=0.5, color='white')),
                hovertemplate='<b>%{customdata[0]}</b><br>' +
                            'Type: %{customdata[2]}<br>' +
                            'Category: %{customdata[1]}<br>' +
                            'Info Type: %{customdata[3]}<br>' +
                            'Position: (%{x:.2f}, %{y:.2f})<br>' +
                            '<extra></extra>',
                customdata=df[['short_name', 'category', 'label', 'info_type']].values
            )
            
            # Add annotations for key entity types
            type_centers = df.groupby('label')[['x', 'y']].mean()
            for entity_type, center in type_centers.iterrows():
                count = len(df[df['label'] == entity_type])
                fig.add_annotation(
                    x=center['x'], y=center['y'],
                    text=f"{entity_type}<br>({count})",
                    showarrow=False,
                    font=dict(size=10, color="black"),
                    bgcolor="rgba(255,255,255,0.8)",
                    bordercolor="gray",
                    borderwidth=1,
                    borderpad=4
                )
            
            # Enhanced layout
            fig.update_layout(
                title={
                    'text': f'NKB Knowledge Graph Entity Embeddings t-SNE{title_suffix}',
                    'x': 0.5,
                    'font': {'size': 16}
                },
                xaxis_title='t-SNE Dimension 1',
                yaxis_title='t-SNE Dimension 2',
                plot_bgcolor='white',
                xaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=1),
                yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=1),
                legend=dict(
                    orientation="v",
                    yanchor="top",
                    y=1,
                    xanchor="left",
                    x=1.02,
                    bgcolor="rgba(255,255,255,0.8)",
                    bordercolor="gray",
                    borderwidth=1
                )
            )
            
            fig.show()
            print("✓ Interactive NKB Plotly visualization displayed")
            
        except Exception as e:
            print(f"❌ Plotly visualization failed: {e}")
            print("📊 Showing matplotlib version instead...")
            show_plotly = False
    
    # Enhanced matplotlib version for NKB
    if not show_plotly or not plotly_available:
        plt.figure(figsize=(16, 12))
        unique_labels = sorted(list(set(labels)))
        
        # Create enhanced color map for NKB
        if nkb_data and 'type_mapping' in nkb_data and 'color_scheme' in nkb_data['type_mapping']:
            nkb_color_scheme = nkb_data['type_mapping']['color_scheme']
            label_to_color = {label: nkb_color_scheme.get(label, '#CCCCCC') for label in unique_labels}
        else:
            # Fallback color scheme
            label_to_color = {}
            for i, label in enumerate(labels):
                if label not in label_to_color:
                    label_to_color[label] = colors[i]
        
        # Plot each entity type with enhanced styling
        for label in unique_labels:
            mask = np.array(labels) == label
            if np.any(mask):
                color = label_to_color[label]
                count = np.sum(mask)
                
                # Different marker sizes based on entity type importance
                marker_size = 30 if label in ['Material', 'Assay', 'Result', 'Publication'] else 20
                alpha = 0.8 if label in ['Material', 'Assay', 'Result', 'Publication'] else 0.6
                
                scatter = plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                                    c=color, label=f'{label} ({count})', 
                                    alpha=alpha, s=marker_size, edgecolors='white', linewidth=0.5)
        
        # Add cluster centers and labels
        type_centers = df.groupby('label')[['x', 'y']].mean()
        for entity_type, center in type_centers.iterrows():
            count = len(df[df['label'] == entity_type])
            plt.annotate(f'{entity_type}\n({count})', 
                        (center['x'], center['y']),
                        xytext=(0, 0), textcoords='offset points',
                        ha='center', va='center',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gray'),
                        fontsize=9, fontweight='bold')
        
        plt.title(f'NKB Knowledge Graph Entity Embeddings t-SNE{title_suffix}', fontsize=18, pad=20)
        plt.xlabel('t-SNE Dimension 1', fontsize=14)
        plt.ylabel('t-SNE Dimension 2', fontsize=14)
        
        # Enhanced legend
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11, 
                  frameon=True, fancybox=True, shadow=True)
        plt.grid(True, alpha=0.3, linestyle='--')
        plt.tight_layout()
        plt.show()
    
    # Print enhanced statistics for NKB
    print(f"✓ NKB t-SNE visualization complete!")
    print(f"📊 Visualized {len(embeddings_2d):,} entities with {len(unique_labels)} different types")
    
    # Show distribution by entity type
    type_distribution = df['label'].value_counts()
    print(f"\n📈 Entity Type Distribution in Visualization:")
    for entity_type, count in type_distribution.items():
        percentage = (count / len(df)) * 100
        print(f"  {entity_type}: {count:,} entities ({percentage:.1f}%)")
    
    # Show category distribution
    category_distribution = df['category'].value_counts()
    print(f"\n🏷️ Entity Category Distribution:")
    for category, count in category_distribution.head(10).items():
        percentage = (count / len(df)) * 100
        print(f"  {category}: {count:,} entities ({percentage:.1f}%)")
    
    return embeddings_2d, df


In [ ]:
# STEP 3: Create NKB-specific visualization (UPDATED)
print("\n🎨 STEP 3: Create NKB Visualization (UPDATED)")
print("="*60)

if 'nkb_data' in locals() and nkb_data:
    # Create visualization using the loaded data
    viz_results = quick_nkb_visualization(sample_size=2000, show_plotly=True)
    
    if viz_results:
        print(f"\n✅ Visualization complete!")
        print(f"  Visualized {len(viz_results['embeddings_2d'])} entities")
        print(f"  Entity types in visualization: {len(set(viz_results['nkb_data']['type_mapping']['entity_labels']))}")
else:
    print("❌ NKB data not loaded. Run Step 2 first.")


In [ ]:
# =============================================================================
# Additional Analysis Functions for NKB Entity Coverage
# =============================================================================

def analyze_nkb_entity_coverage(nkb_data, expected_entity_types):
    """
    Analyze coverage of expected entity types from your previous analysis
    """
    if not nkb_data:
        print("❌ No NKB data available")
        return
    
    print("🔍 Analyzing NKB Entity Coverage...")
    print("="*50)
    
    # Expected entity types from your previous analysis
    expected_types = {
        'Material': ['NPO_199', 'MaterialID', 'CoreComposition', 'ShellComposition'],
        'Assay': ['BAO_0000179', 'AssayID', 'AssayType', 'AssayProtocol'],
        'Result': ['NPO_1680', 'ResultID', 'ResultValue', 'ResultUnit'],
        'Publication': ['OBI_0000070', 'publication_DOI', 'PublicationYear'],
        'Medium': ['MediumID', 'MediumType', 'MediumComposition'],
        'Parameter': ['ParameterID', 'ParameterName', 'ParameterValue'],
        'MaterialFG': ['MaterialFGID', 'FunctionalGroup'],
        'Additive': ['AdditiveID', 'AdditiveType', 'AdditiveConcentration'],
        'Contamination': ['ContaminationID', 'ContaminationType'],
        'MolecularResult': ['MolecularResultID', 'MolecularData']
    }
    
    id_to_entity = nkb_data['id_to_entity']
    type_mapping = nkb_data['type_mapping']
    
    print(f"📊 Entity Type Coverage Analysis:")
    print(f"Total entities in embeddings: {len(id_to_entity):,}")
    
    # Check coverage for each expected type
    for expected_type, patterns in expected_types.items():
        found_entities = []
        
        for entity_id, entity_uri in id_to_entity.items():
            entity_lower = entity_uri.lower()
            
            # Check if entity matches any pattern for this type
            if any(pattern.lower() in entity_lower for pattern in patterns):
                found_entities.append(entity_uri)
        
        coverage_percent = (len(found_entities) / len(id_to_entity)) * 100
        print(f"  {expected_type}: {len(found_entities):,} entities ({coverage_percent:.2f}%)")
        
        # Show examples
        if found_entities:
            print(f"    Examples: {[entity.split('/')[-1].split('#')[-1][:30] for entity in found_entities[:3]]}")
        else:
            print(f"    ❌ No entities found matching patterns: {patterns}")
    
    # Show actual type distribution from our mapping
    print(f"\n📈 Actual Type Distribution (from mapping):")
    type_counts = type_mapping['type_counts']
    for entity_type, count in type_counts.most_common():
        percentage = (count / len(id_to_entity)) * 100
        print(f"  {entity_type}: {count:,} entities ({percentage:.1f}%)")
    
    return {
        'total_entities': len(id_to_entity),
        'type_distribution': type_counts,
        'coverage_analysis': expected_types
    }

def check_specific_entities(nkb_data, entity_patterns):
    """
    Check if specific entities or patterns exist in the embeddings
    """
    if not nkb_data:
        print("❌ No NKB data available")
        return
    
    print("🔍 Checking Specific Entity Patterns...")
    print("="*50)
    
    id_to_entity = nkb_data['id_to_entity']
    
    for pattern_name, pattern in entity_patterns.items():
        matching_entities = []
        
        for entity_id, entity_uri in id_to_entity.items():
            if pattern.lower() in entity_uri.lower():
                matching_entities.append((entity_id, entity_uri))
        
        print(f"Pattern '{pattern_name}' ('{pattern}'):")
        print(f"  Found {len(matching_entities)} matches")
        
        if matching_entities:
            print(f"  Examples:")
            for i, (entity_id, entity_uri) in enumerate(matching_entities[:5]):
                short_uri = entity_uri.split('/')[-1].split('#')[-1][:50]
                print(f"    {i+1}. ID {entity_id}: {short_uri}")
        else:
            print(f"  ❌ No matches found")
        print()
    
    return matching_entities

# Example usage - check for specific patterns from your original analysis
print("🔍 STEP 4: Check Entity Coverage")
print("="*50)

if 'nkb_data' in locals() and nkb_data:
    # Check coverage of expected entity types
    coverage_results = analyze_nkb_entity_coverage(nkb_data, {})
    
    print("\n" + "="*50)
    print("🔍 STEP 5: Check Specific Entity Patterns")
    print("="*50)
    
    # Check for specific entity patterns from your analysis
    specific_patterns = {
        'Material_IDs': 'MaterialID',
        'Publication_DOIs': 'publication_DOI', 
        'Assay_IDs': 'AssayID',
        'Result_IDs': 'ResultID',
        'NPO_199_Materials': 'NPO_199',
        'NPO_1680_Results': 'NPO_1680',
        'BAO_Assays': 'BAO_0000179',
        'OBI_Publications': 'OBI_0000070',
        'Generated_IDs': 'n7656432b49624fcfa673c580ed678c01b'
    }
    
    pattern_results = check_specific_entities(nkb_data, specific_patterns)
else:
    print("❌ NKB data not loaded. Run previous steps first.")


In [3]:
#!/usr/bin/env python3
"""
RDF Graph Parser and Analyzer
Analyzes any RDF/TTL file to discover entity types and relations
"""

import rdflib
from collections import Counter, defaultdict
import pandas as pd
import json
from pathlib import Path

def parse_rdf_comprehensive(ttl_file="mappings/NKB_RDF_V3.ttl", max_examples=10):
    """
    Comprehensive RDF parsing to discover all entity types and relations
    """
    print(f"🔍 Analyzing RDF file: {ttl_file}")
    print("="*60)
    
    try:
        # Load the RDF graph
        g = rdflib.Graph()
        print("Loading graph...")
        g.parse(ttl_file, format="turtle")
        print(f"✅ Successfully loaded {len(g):,} triples")
        
    except Exception as e:
        print(f"❌ Error loading RDF file: {e}")
        return None
    
    # Initialize data structures
    predicates = Counter()
    subjects = set()
    objects = set()
    entity_types = defaultdict(set)
    type_counts = Counter()
    literal_predicates = set()
    uri_predicates = set()
    
    # Potential type predicates to look for
    type_predicate_patterns = [
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
        "http://www.w3.org/2000/01/rdf-schema#type",
        "rdf:type",
        "rdfs:type",
        "a"  # Turtle shorthand for rdf:type
    ]
    
    print("\n📊 Analyzing triples...")
    
    # First pass: collect all predicates and categorize them
    for i, (s, p, o) in enumerate(g):
        if i % 100000 == 0 and i > 0:
            print(f"  Processed {i:,} triples...")
            
        subj_str = str(s)
        pred_str = str(p) 
        obj_str = str(o)
        
        # Count predicates
        predicates[pred_str] += 1
        
        # Collect unique subjects and objects (entities)
        subjects.add(subj_str)
        if isinstance(o, rdflib.URIRef):
            objects.add(obj_str)
            uri_predicates.add(pred_str)
        else:
            literal_predicates.add(pred_str)
        
        # Look for type information
        is_type_predicate = (
            pred_str in type_predicate_patterns or
            'type' in pred_str.lower() or
            pred_str.endswith('#type') or
            pred_str.endswith('/type')
        )
        
        if is_type_predicate:
            entity_types[subj_str].add(obj_str)
            type_counts[obj_str] += 1
    
    # Calculate statistics
    all_entities = subjects.union(objects)
    
    print(f"\n📈 Basic Statistics:")
    print(f"  Total triples: {len(g):,}")
    print(f"  Unique subjects: {len(subjects):,}")
    print(f"  Unique objects: {len(objects):,}")
    print(f"  Total unique entities: {len(all_entities):,}")
    print(f"  Unique predicates: {len(predicates):,}")
    print(f"  Predicates with URI objects: {len(uri_predicates):,}")
    print(f"  Predicates with literal objects: {len(literal_predicates):,}")
    
    return analyze_results(g, predicates, entity_types, type_counts, 
                          uri_predicates, literal_predicates, all_entities, max_examples)

def analyze_results(g, predicates, entity_types, type_counts, 
                   uri_predicates, literal_predicates, all_entities, max_examples):
    """Analyze and display the parsing results"""
    
    # 1. PREDICATE ANALYSIS
    print(f"\n🔗 PREDICATE ANALYSIS")
    print("="*60)
    print(f"Found {len(predicates)} unique predicates:")
    
    print(f"\n📊 Top {min(20, len(predicates))} most frequent predicates:")
    for i, (pred, count) in enumerate(predicates.most_common(20), 1):
        pred_name = extract_local_name(pred)
        pred_type = "🔗 URI" if pred in uri_predicates else "📝 Literal"
        print(f"  {i:2d}. {pred_type} {pred_name:<30} ({count:,} times)")
    
    # Show some examples for top predicates
    print(f"\n🔍 Examples for top 5 predicates:")
    for pred, count in list(predicates.most_common(5)):
        pred_name = extract_local_name(pred)
        print(f"\n  {pred_name} ({count:,} occurrences):")
        
        examples = []
        for s, p, o in g:
            if str(p) == pred and len(examples) < max_examples:
                subj_name = extract_local_name(str(s))
                obj_name = extract_local_name(str(o))
                examples.append(f"    {subj_name} → {obj_name}")
        
        for example in examples:
            print(example)
    
    # 2. ENTITY TYPE ANALYSIS
    print(f"\n🏷️  ENTITY TYPE ANALYSIS")
    print("="*60)
    
    if len(type_counts) > 0:
        print(f"Found {len(type_counts)} distinct entity types:")
        print(f"\n📊 Entity type distribution:")
        
        for i, (entity_type, count) in enumerate(type_counts.most_common(15), 1):
            type_name = extract_local_name(entity_type)
            print(f"  {i:2d}. {type_name:<30} ({count:,} entities)")
        
        # Show examples of entities for each type
        print(f"\n🔍 Example entities by type (top 5 types):")
        for entity_type, count in list(type_counts.most_common(5)):
            type_name = extract_local_name(entity_type)
            print(f"\n  {type_name} ({count:,} entities):")
            
            # Find entities of this type
            examples = []
            for entity, types in entity_types.items():
                if entity_type in types and len(examples) < max_examples:
                    entity_name = extract_local_name(entity)
                    examples.append(f"    {entity_name}")
            
            for example in examples:
                print(example)
    else:
        print("❌ No explicit entity types found using standard type predicates")
        print("💡 This could mean:")
        print("  - The RDF uses non-standard type predicates")
        print("  - Entity types are encoded in URI patterns")
        print("  - The graph uses a different schema")
        
        # Try to infer types from URI patterns
        print(f"\n🔤 Attempting to infer types from URI patterns...")
        infer_types_from_uris(all_entities, max_examples)
    
    # 3. RELATION ANALYSIS (URI predicates only)
    print(f"\n🔗 RELATION ANALYSIS (Entity-to-Entity)")
    print("="*60)
    
    uri_pred_counts = {p: c for p, c in predicates.items() if p in uri_predicates}
    print(f"Found {len(uri_pred_counts)} predicates connecting entities:")
    
    for i, (pred, count) in enumerate(Counter(uri_pred_counts).most_common(15), 1):
        pred_name = extract_local_name(pred)
        print(f"  {i:2d}. {pred_name:<30} ({count:,} connections)")
    
    # 4. SUMMARY AND RECOMMENDATIONS
    print(f"\n📋 SUMMARY")
    print("="*60)
    print(f"Your knowledge graph contains:")
    print(f"  • {len(all_entities):,} unique entities")
    print(f"  • {len(uri_pred_counts):,} types of relations between entities")
    print(f"  • {len(type_counts):,} explicit entity types" + 
          (" (good!)" if len(type_counts) > 0 else " (may need inference)"))
    print(f"  • {len(literal_predicates):,} attribute predicates (entity→literal)")
    
    return {
        'predicates': predicates,
        'entity_types': dict(entity_types),
        'type_counts': type_counts,
        'uri_predicates': uri_pred_counts,
        'literal_predicates': {p: predicates[p] for p in literal_predicates},
        'all_entities': all_entities,
        'graph': g
    }

def extract_local_name(uri):
    """Extract the local name from a URI"""
    if '#' in uri:
        return uri.split('#')[-1]
    elif '/' in uri:
        return uri.split('/')[-1]
    else:
        return uri

def infer_types_from_uris(entities, max_examples=5):
    """Try to infer entity types from URI patterns"""
    
    # Common URI patterns that might indicate types
    patterns = {
        'Person': ['person', 'people', 'author', 'researcher', 'human'],
        'Organization': ['org', 'organization', 'company', 'institution'],
        'Publication': ['paper', 'article', 'publication', 'document'],
        'Concept': ['concept', 'topic', 'subject', 'category'],
        'Location': ['place', 'location', 'city', 'country'],
        'Event': ['event', 'conference', 'meeting'],
        'Dataset': ['dataset', 'data', 'corpus'],
        'Resource': ['resource', 'material', 'source']
    }
    
    inferred_types = defaultdict(list)
    
    for entity in list(entities)[:10000]:  # Sample for performance
        entity_lower = entity.lower()
        
        for type_name, keywords in patterns.items():
            if any(keyword in entity_lower for keyword in keywords):
                inferred_types[type_name].append(entity)
                break
    
    if inferred_types:
        print("🔤 Inferred types from URI patterns:")
        for type_name, entities_list in inferred_types.items():
            print(f"\n  {type_name} ({len(entities_list)} entities):")
            for entity in entities_list[:max_examples]:
                print(f"    {extract_local_name(entity)}")
    else:
        print("❌ Could not infer types from URI patterns")

def save_analysis_results(results, output_file="rdf_analysis.json"):
    """Save analysis results to a JSON file"""
    
    # Convert to serializable format
    serializable_results = {
        'predicates': dict(results['predicates']),
        'type_counts': dict(results['type_counts']),
        'uri_predicates': results['uri_predicates'],
        'literal_predicates': results['literal_predicates'],
        'num_entities': len(results['all_entities']),
        'num_triples': len(results['graph'])
    }
    
    with open(output_file, 'w') as f:
        json.dump(serializable_results, f, indent=2)
    
    print(f"\n💾 Analysis results saved to: {output_file}")

def create_analysis_dataframes(results):
    """Create pandas DataFrames for easier analysis"""
    
    # Predicates DataFrame
    pred_df = pd.DataFrame([
        {
            'predicate': pred,
            'local_name': extract_local_name(pred),
            'count': count,
            'type': 'relation' if pred in results['uri_predicates'] else 'attribute'
        }
        for pred, count in results['predicates'].items()
    ]).sort_values('count', ascending=False)
    
    # Entity Types DataFrame
    if results['type_counts']:
        types_df = pd.DataFrame([
            {
                'entity_type': etype,
                'local_name': extract_local_name(etype),
                'count': count
            }
            for etype, count in results['type_counts'].items()
        ]).sort_values('count', ascending=False)
    else:
        types_df = pd.DataFrame()
    
    return pred_df, types_df

def main():
    """Main function to run the analysis"""
    
    # Change this to your RDF file path
    ttl_file = "mappings/NKB_RDF_V3.ttl"
    
    if not Path(ttl_file).exists():
        print(f"❌ File not found: {ttl_file}")
        print("Please update the file path in the script")
        return
    
    # Run the analysis
    results = parse_rdf_comprehensive(ttl_file, max_examples=5)
    
    if results:
        # Create DataFrames for easier manipulation
        pred_df, types_df = create_analysis_dataframes(results)
        
        print(f"\n📊 DataFrames created for further analysis:")
        print(f"  • pred_df: {len(pred_df)} predicates")
        print(f"  • types_df: {len(types_df)} entity types")
        
        # Save results
        save_analysis_results(results)
        
        # Show how to access the data
        print(f"\n💡 To explore further:")
        print(f"  results['predicates']     # All predicates with counts")
        print(f"  results['type_counts']    # Entity types with counts") 
        print(f"  results['uri_predicates'] # Relations between entities")
        print(f"  pred_df                   # Predicates as DataFrame")
        print(f"  types_df                  # Entity types as DataFrame")
        
        return results, pred_df, types_df
    
    return None

if __name__ == "__main__":
    results, pred_df, types_df = main()

🔍 Analyzing RDF file: mappings/NKB_RDF_V3.ttl
Loading graph...
✅ Successfully loaded 1,369,675 triples

📊 Analyzing triples...
  Processed 100,000 triples...
  Processed 200,000 triples...
  Processed 300,000 triples...
  Processed 400,000 triples...
  Processed 500,000 triples...
  Processed 600,000 triples...
  Processed 700,000 triples...
  Processed 800,000 triples...
  Processed 900,000 triples...
  Processed 1,000,000 triples...
  Processed 1,100,000 triples...
  Processed 1,200,000 triples...
  Processed 1,300,000 triples...

📈 Basic Statistics:
  Total triples: 1,369,675
  Unique subjects: 290,789
  Unique objects: 1,124
  Total unique entities: 291,170
  Unique predicates: 71
  Predicates with URI objects: 24
  Predicates with literal objects: 68

🔗 PREDICATE ANALYSIS
Found 71 unique predicates:

📊 Top 20 most frequent predicates:
   1. 📝 Literal identifier                     (284,602 times)
   2. 📝 Literal OBI_0002110                    (176,420 times)
   3. 🔗 URI type      

In [ ]:
# STEP 3: Enhanced NKB t-SNE Visualization (FIXED)
print("="*60)
print("STEP 3: Enhanced NKB t-SNE Visualization (FIXED)")
print("="*60)
print("Creating enhanced NKB-specific t-SNE visualization...")

# Get the loaded data
entity_embeddings = nkb_data['entity_embeddings']
id_to_entity = nkb_data['id_to_entity']
type_mapping = nkb_data['type_mapping']

entity_colors = type_mapping['entity_colors']
entity_labels = type_mapping['entity_labels']

# Create enhanced visualization with NKB-specific features
try:
    embeddings_2d, enhanced_df = create_nkb_tsne_visualization(
        entity_embeddings, id_to_entity, entity_colors, entity_labels,
        nkb_data=nkb_data, sample_size=3000, 
        title_suffix=" - Enhanced NKB Analysis", 
        show_plotly=True
    )

    if embeddings_2d is not None:
        print(f"\n✅ Enhanced NKB visualization complete!")
        print(f"📊 Visualization Summary:")
        print(f"  - Visualized entities: {len(embeddings_2d):,}")
        print(f"  - Entity types: {len(set(entity_labels))}")
        print(f"  - Embedding dimensions: {entity_embeddings.shape}")
        
        # Show type distribution in visualization
        if enhanced_df is not None:
            type_counts = enhanced_df['label'].value_counts()
            print(f"\n📈 Top entity types in visualization:")
            for entity_type, count in type_counts.head(10).items():
                percentage = (count / len(enhanced_df)) * 100
                print(f"    {entity_type}: {count:,} ({percentage:.1f}%)")
    else:
        print("❌ Visualization failed")
        
except Exception as e:
    print(f"❌ Error in visualization: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# Embedding Visualization for Jupyter Notebooks
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import rdflib
from collections import Counter, defaultdict
import networkx as nx

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# =============================================================================
# Data Loading Functions
# =============================================================================

def load_embeddings(entity_path="complex_embeddings/complex_entity_embeddings.pt", 
                   relation_path="complex_embeddings/complex_relation_embeddings.pt",
                   entity_to_id_path="complex_embeddings/complex_entity_to_id.pt", 
                   relation_to_id_path="complex_embeddings/complex_relation_to_id.pt"):
    """Load all embedding files and create lookup dictionaries"""
    try:
        entity_embeddings = torch.load(entity_path).detach().cpu().numpy()
        relation_embeddings = torch.load(relation_path).detach().cpu().numpy()
        entity_to_id = torch.load(entity_to_id_path)
        relation_to_id = torch.load(relation_to_id_path)
        
        # Create reverse lookup dictionaries from the ones we have
        id_to_entity = {v: k for k, v in entity_to_id.items()}
        id_to_relation = {v: k for k, v in relation_to_id.items()}
        
        print(f"✓ Loaded embeddings:")
        print(f"  Entities: {entity_embeddings.shape} ({len(entity_to_id)} unique)")
        print(f"  Relations: {relation_embeddings.shape} ({len(relation_to_id)} unique)")
        print(f"  Sample entities: {list(entity_to_id.keys())[:3]}...")
        print(f"  Sample relations: {list(relation_to_id.keys())[:3]}...")
        
        return entity_embeddings, relation_embeddings, entity_to_id, relation_to_id, id_to_entity, id_to_relation
        
    except Exception as e:
        print(f"❌ Error loading embeddings: {e}")
        print("Make sure these files exist:")
        print(f"  - {entity_path}")
        print(f"  - {relation_path}")
        print(f"  - {entity_to_id_path}")
        print(f"  - {relation_to_id_path}")
        return None, None, None, None, None, None

# =============================================================================
# Entity Type Analysis Functions
# =============================================================================

# =============================================================================
# Debug Functions for TTL Analysis
# =============================================================================

def debug_ttl_file(ttl_file="oreganov2.1_metadata_complete.ttl", max_triples=50):
    """Debug TTL file to see what's actually in there"""
    print(f"🔍 Debugging TTL file: {ttl_file}")
    
    try:
        g = rdflib.Graph()
        print("Loading graph...")
        g.parse(ttl_file, format="turtle")
        print(f"✓ Successfully loaded graph with {len(g)} triples")
        
        # Show first few triples
        print(f"\n📋 First {min(max_triples, len(g))} triples:")
        for i, (s, p, o) in enumerate(g):
            if i >= max_triples:
                break
            print(f"  {i+1}. Subject: {str(s)[:60]}...")
            print(f"     Predicate: {str(p)}")
            print(f"     Object: {str(o)[:60]}...")
            print()
        
        # Count different predicates
        predicates = Counter()
        type_predicates = []
        
        print("🔗 Analyzing predicates...")
        for s, p, o in g:
            pred_str = str(p)
            predicates[pred_str] += 1
            
            # Look for type-related predicates (more flexible)
            if 'type' in pred_str.lower():
                type_predicates.append(pred_str)
        
        print(f"\n📊 Most common predicates:")
        for pred, count in predicates.most_common(15):
            short_pred = pred.split('/')[-1].split('#')[-1] if '/' in pred or '#' in pred else pred
            print(f"  {short_pred}: {count:,} occurrences")
        
        # Show type-related predicates
        unique_type_preds = list(set(type_predicates))
        if unique_type_preds:
            print(f"\n🏷️  Found {len(unique_type_preds)} type-related predicates:")
            for pred in unique_type_preds[:10]:
                print(f"  {pred}")
        else:
            print(f"\n❌ No predicates containing 'type' found")
        
        # Try different type predicate patterns
        type_patterns = [
            "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
            "http://www.w3.org/2000/01/rdf-schema#type", 
            "rdf:type",
            "rdfs:type",
            "a"  # shorthand for rdf:type in turtle
        ]
        
        print(f"\n🔍 Testing different type patterns:")
        for pattern in type_patterns:
            count = sum(1 for s, p, o in g if str(p) == pattern)
            print(f"  '{pattern}': {count} matches")
        
        return g, predicates
        
    except Exception as e:
        print(f"❌ Error loading TTL file: {e}")
        return None, None

def analyze_entity_types_flexible(ttl_file="oreganov2.1_metadata_complete.ttl"):
    """More flexible entity type analysis"""
    print("📊 Analyzing entity types (flexible approach)...")
    
    entity_types = defaultdict(set)
    type_counts = Counter()
    
    try:
        g = rdflib.Graph()
        g.parse(ttl_file, format="turtle")
        print(f"✓ Loaded {len(g)} triples")
        
        # Try multiple type predicate patterns
        type_patterns = [
            "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
            "http://www.w3.org/2000/01/rdf-schema#type",
            "rdf:type",
            "rdfs:type"
        ]
        
        total_type_triples = 0
        
        for s, p, o in g:
            pred_str = str(p)
            
            # Check if this is a type predicate (flexible matching)
            is_type_predicate = False
            
            # Exact matches
            if pred_str in type_patterns:
                is_type_predicate = True
            # Contains 'type' and looks like a type predicate
            elif 'type' in pred_str.lower() and ('rdf' in pred_str or 'rdfs' in pred_str):
                is_type_predicate = True
            # Turtle shorthand 'a' for rdf:type
            elif pred_str == 'a':
                is_type_predicate = True
            
            if is_type_predicate:
                entity = str(s)
                entity_type = str(o)
                entity_types[entity].add(entity_type)
                type_counts[entity_type] += 1
                total_type_triples += 1
        
        print(f"Found {total_type_triples} type triples")
        
        if len(type_counts) > 0:
            print(f"Found {len(type_counts)} different entity types:")
            for entity_type, count in type_counts.most_common(15):
                short_type = entity_type.split('/')[-1].split('#')[-1]
                print(f"  {short_type}: {count:,} entities")
        else:
            print("❌ Still no entity types found")
            
            # Let's try looking for any triples where the object looks like a type/class
            print("\n🔍 Looking for class-like objects...")
            class_like_objects = Counter()
            
            for s, p, o in g:
                obj_str = str(o)
                # Look for objects that might be classes/types
                if any(indicator in obj_str.lower() for indicator in ['class', 'type', 'concept', 'category']):
                    class_like_objects[obj_str] += 1
                # Look for objects with common ontology patterns
                elif any(pattern in obj_str for pattern in ['#', '/', 'owl:', 'rdfs:', 'schema:']):
                    if not obj_str.startswith('http://www.w3.org/2001/XMLSchema'):  # Skip XSD types
                        class_like_objects[obj_str] += 1
            
            if class_like_objects:
                print("Found potential class/type objects:")
                for obj, count in class_like_objects.most_common(10):
                    short_obj = obj.split('/')[-1].split('#')[-1]
                    print(f"  {short_obj}: {count} occurrences")
        
        return entity_types, type_counts
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return defaultdict(set), Counter()
def analyze_entity_types(ttl_file="oreganov2.1_metadata_complete.ttl"):
    """Analyze entity types from TTL file or infer from entity names"""
    print("📊 Analyzing entity types...")
    
    entity_types = defaultdict(set)
    type_counts = Counter()
    
    # Try to load TTL file if it exists
    try:
        g = rdflib.Graph()
        g.parse(ttl_file, format="turtle")
        
        for s, p, o in g:
            if str(p) == "http://www.w3.org/1999/02/22-rdf-syntax-ns#type":
                entity = str(s)
                entity_type = str(o)
                entity_types[entity].add(entity_type)
                type_counts[entity_type] += 1
        
        if len(type_counts) > 0:
            print(f"Found {len(type_counts)} different entity types from TTL:")
            for entity_type, count in type_counts.most_common(10):
                short_type = entity_type.split('/')[-1].split('#')[-1]
                print(f"  {short_type}: {count:,} entities")
        else:
            print("No entity types found in TTL file, will infer from names")
            
    except Exception as e:
        print(f"Could not load TTL file ({e})")
        print("Will infer entity types from entity names instead")
    
    return entity_types, type_counts

def infer_entity_types_from_names(id_to_entity):
    """Infer entity types from entity names/URIs when TTL is not available"""
    print("🔍 Inferring entity types from entity names...")
    
    type_counts = Counter()
    entity_types = {}
    
    # Common patterns to look for in entity names/URIs
    type_patterns = {
        'Drug': ['drug', 'compound', 'medication', 'pharmaceutical', 'chemical'],
        'Gene': ['gene', 'dna', 'genetic'],
        'Protein': ['protein', 'enzyme', 'receptor'],
        'Disease': ['disease', 'disorder', 'syndrome', 'condition', 'illness'],
        'Phenotype': ['phenotype', 'trait', 'characteristic'],
        'Target': ['target', 'binding'],
        'Indication': ['indication', 'treatment'],
        'SideEffect': ['side_effect', 'adverse', 'reaction'],
        'Activity': ['activity', 'function', 'mechanism'],
        'Pathway': ['pathway', 'process', 'signaling']
    }
    
    for entity_id, entity_name in id_to_entity.items():
        entity_lower = entity_name.lower()
        entity_type = 'Unknown'
        
        # Check each pattern
        for type_name, patterns in type_patterns.items():
            if any(pattern in entity_lower for pattern in patterns):
                entity_type = type_name
                break
        
        entity_types[entity_name] = {entity_type}
        type_counts[entity_type] += 1
    
    print(f"Inferred types for {len(entity_types)} entities:")
    for entity_type, count in type_counts.most_common():
        print(f"  {entity_type}: {count:,} entities")
    
    return entity_types, type_counts

def get_entity_colors_by_type(entity_types, id_to_entity, use_name_inference=True):
    """Assign colors based on entity types"""
    type_colors = {
        'Drug': '#FF6B6B',           # Red
        'Gene': '#4ECDC4',           # Teal  
        'Protein': '#45B7D1',        # Blue
        'Disease': '#96CEB4',        # Green
        'Phenotype': '#FFEAA7',      # Yellow
        'Compound': '#DDA0DD',       # Plum
        'Target': '#FFA07A',         # Light salmon
        'Indication': '#98D8C8',     # Mint
        'SideEffect': '#F7DC6F',     # Light yellow
        'Activity': '#BB8FCE',       # Light purple
        'Pathway': '#85C1E9',        # Light blue
        'Unknown': '#CCCCCC'         # Gray
    }
    
    entity_colors = []
    entity_labels = []
    
    # If no entity types were found from TTL, infer from names
    if len(entity_types) == 0 and use_name_inference:
        print("🔄 No TTL types found, inferring from entity names...")
        entity_types, _ = infer_entity_types_from_names(id_to_entity)
    
    for entity_id in range(len(id_to_entity)):
        entity = id_to_entity[entity_id]
        
        entity_type = 'Unknown'
        color = type_colors['Unknown']
        
        if entity in entity_types:
            types = entity_types[entity]
            for t in types:
                # Handle both full URIs and simple type names
                type_name = t.split('/')[-1].split('#')[-1] if isinstance(t, str) else str(t)
                if type_name in type_colors:
                    entity_type = type_name
                    color = type_colors[type_name]
                    break
        else:
            # Fallback: infer type from URI patterns
            entity_lower = entity.lower()
            if 'drug' in entity_lower or 'compound' in entity_lower:
                entity_type = 'Drug'
                color = type_colors['Drug']
            elif 'gene' in entity_lower:
                entity_type = 'Gene' 
                color = type_colors['Gene']
            elif 'protein' in entity_lower:
                entity_type = 'Protein'
                color = type_colors['Protein']
            elif 'disease' in entity_lower:
                entity_type = 'Disease'
                color = type_colors['Disease']
            elif 'phenotype' in entity_lower:
                entity_type = 'Phenotype'
                color = type_colors['Phenotype']
        
        entity_colors.append(color)
        entity_labels.append(entity_type)
    
    return entity_colors, entity_labels

def get_entity_colors_by_relation_involvement(id_to_entity, ttl_file="oreganov2.1_metadata_complete.ttl"):
    """Color entities by which relations they participate in most"""
    print("🔗 Analyzing entity-relation involvement...")
    
    g = rdflib.Graph()
    g.parse(ttl_file, format="turtle")
    
    entity_relation_counts = defaultdict(lambda: defaultdict(int))
    
    for s, p, o in g:
        if isinstance(o, rdflib.URIRef):
            head = str(s)
            relation = str(p)
            tail = str(o)
            
            entity_relation_counts[head][relation] += 1
            entity_relation_counts[tail][relation] += 1
    
    relation_colors = {
        'acts_within': '#FF6B6B',
        'has_phenotype': '#4ECDC4', 
        'has_side_effect': '#45B7D1',
        'has_target': '#96CEB4',
        'has_indication': '#FFEAA7',
        'has_activity': '#DDA0DD',
        'causes_condition': '#FFA07A',
        'has_effect': '#98D8C8',
        'gene_product_of': '#F7DC6F',
        'type': '#CCCCCC'
    }
    
    entity_colors = []
    entity_labels = []
    
    for entity_id in range(len(id_to_entity)):
        entity = id_to_entity[entity_id]
        
        if entity in entity_relation_counts:
            relation_counts = entity_relation_counts[entity]
            dominant_relation = max(relation_counts.items(), key=lambda x: x[1])[0]
            
            rel_name = dominant_relation.split('#')[-1] if '#' in dominant_relation else dominant_relation.split('/')[-1]
            
            color = relation_colors.get(rel_name, '#CCCCCC')
            entity_colors.append(color)
            entity_labels.append(rel_name)
        else:
            entity_colors.append('#CCCCCC')
            entity_labels.append('None')
    
    return entity_colors, entity_labels

# =============================================================================
# Visualization Functions
# =============================================================================

def create_tsne_visualization(entity_embeddings, id_to_entity, entity_colors, entity_labels, 
                             sample_size=5000, title_suffix="", save_prefix="entity_tsne", show_plotly=False):
    """Create t-SNE visualization with colored points"""
    print(f"🎨 Creating t-SNE visualization...")
    
    # Sample entities if too many
    if len(entity_embeddings) > sample_size:
        indices = np.random.choice(len(entity_embeddings), sample_size, replace=False)
        sampled_embeddings = entity_embeddings[indices]
        colors = [entity_colors[i] for i in indices]
        labels = [entity_labels[i] for i in indices]
        entity_names = [id_to_entity[i] for i in indices]
    else:
        indices = np.arange(len(entity_embeddings))
        sampled_embeddings = entity_embeddings
        colors = entity_colors
        labels = entity_labels
        entity_names = [id_to_entity[i] for i in indices]
    
    # Apply t-SNE
    print("Applying t-SNE...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(sampled_embeddings)-1))
    embeddings_2d = tsne.fit_transform(sampled_embeddings)
    
    # Create DataFrame for easy handling
    df = pd.DataFrame({
        'x': embeddings_2d[:, 0],
        'y': embeddings_2d[:, 1], 
        'color': colors,
        'label': labels,
        'entity': [name[:50] + '...' if len(name) > 50 else name for name in entity_names]
    })
    
    # Try interactive Plotly plot (optional)
    if show_plotly:
        try:
            fig = px.scatter(df, x='x', y='y', color='label', hover_data=['entity'],
                            title=f'Entity Embeddings t-SNE{title_suffix}',
                            width=1000, height=800)
            
            fig.update_traces(marker=dict(size=3, opacity=0.7))
            fig.show()
            print("✓ Interactive Plotly plot displayed")
        except Exception as e:
            print(f"❌ Plotly visualization failed: {e}")
            print("📊 Showing matplotlib version instead...")
    
    # Create matplotlib version (always works)
    plt.figure(figsize=(15, 10))
    unique_labels = list(set(labels))
    
    # Create a color map for consistent colors
    label_to_color = {}
    for i, label in enumerate(labels):
        if label not in label_to_color:
            label_to_color[label] = colors[i]
    
    for label in unique_labels:
        mask = np.array(labels) == label
        if np.any(mask):
            color = label_to_color[label]
            plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                       c=color, label=label, alpha=0.6, s=20)
    
    plt.title(f'Entity Embeddings t-SNE{title_suffix}', fontsize=16)
    plt.xlabel('t-SNE Dimension 1', fontsize=12)
    plt.ylabel('t-SNE Dimension 2', fontsize=12)
    
    # Better legend positioning
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Matplotlib visualization complete!")
    print(f"📊 Visualized {len(embeddings_2d)} entities with {len(unique_labels)} different types")
    
    return embeddings_2d, df

def create_relation_visualization(relation_embeddings, id_to_relation):
    """Create visualization of relation embeddings"""
    print("🔗 Creating relation embedding visualization...")
    
    # Apply PCA for relations
    pca = PCA(n_components=2)
    relation_2d = pca.fit_transform(relation_embeddings)
    
    # Create relation names
    relation_names = [id_to_relation[i].split('#')[-1] if '#' in id_to_relation[i] 
                     else id_to_relation[i].split('/')[-1] for i in range(len(id_to_relation))]
    
    # Semantic groups for coloring
    semantic_groups = {
        'effect': ['has_effect', 'has_side_effect', 'increase_effect', 'decrease_effect'],
        'activity': ['has_activity', 'increase_activity', 'decrease_activity'],
        'efficacy': ['increase_efficacy', 'decrease_efficacy'],
        'medical': ['has_indication', 'causes_condition', 'is_substance_that_treats'],
        'biological': ['has_target', 'gene_product_of', 'has_phenotype'],
        'location': ['acts_within'],
        'general': ['type', 'is_affecting']
    }
    
    group_colors = {
        'effect': '#FF6B6B',
        'activity': '#4ECDC4',
        'efficacy': '#45B7D1', 
        'medical': '#96CEB4',
        'biological': '#FFEAA7',
        'location': '#DDA0DD',
        'general': '#CCCCCC'
    }
    
    relation_groups = []
    for name in relation_names:
        group = 'general'
        for g, relations in semantic_groups.items():
            if any(rel in name.lower() for rel in relations):
                group = g
                break
        relation_groups.append(group)
    
    # Create plot
    plt.figure(figsize=(12, 8))
    for group in set(relation_groups):
        mask = np.array(relation_groups) == group
        plt.scatter(relation_2d[mask, 0], relation_2d[mask, 1], 
                   c=group_colors[group], label=group, s=100, alpha=0.8)
    
    # Add relation labels
    for i, name in enumerate(relation_names):
        plt.annotate(name, (relation_2d[i, 0], relation_2d[i, 1]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.title('Relation Embeddings (PCA)')
    plt.xlabel('PC 1')
    plt.ylabel('PC 2')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return relation_2d, relation_names, relation_groups

def create_3d_visualization(entity_embeddings, id_to_entity, entity_colors, entity_labels, sample_size=3000, show_plotly=False):
    """Create 3D visualization of embeddings"""
    print("🌐 Creating 3D visualization...")
    
    # Sample entities
    if len(entity_embeddings) > sample_size:
        indices = np.random.choice(len(entity_embeddings), sample_size, replace=False)
        sampled_embeddings = entity_embeddings[indices]
        colors = [entity_colors[i] for i in indices]
        labels = [entity_labels[i] for i in indices]
        entity_names = [id_to_entity[i] for i in indices]
    else:
        indices = np.arange(len(entity_embeddings))
        sampled_embeddings = entity_embeddings
        colors = entity_colors
        labels = entity_labels
        entity_names = [id_to_entity[i] for i in indices]
    
    # Apply t-SNE for 3D
    tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(30, len(sampled_embeddings)-1))
    embeddings_3d = tsne_3d.fit_transform(sampled_embeddings)
    
    if show_plotly:
        try:
            # Create 3D plot
            fig = go.Figure(data=[go.Scatter3d(
                x=embeddings_3d[:, 0],
                y=embeddings_3d[:, 1],
                z=embeddings_3d[:, 2],
                mode='markers',
                marker=dict(
                    size=3,
                    color=labels,
                    colorscale='Viridis',
                    opacity=0.7
                ),
                text=[name[:50] for name in entity_names],
                hovertemplate='<b>%{text}</b><br>Type: %{marker.color}<extra></extra>'
            )])
            
            fig.update_layout(
                title='3D Entity Embeddings Visualization',
                scene=dict(
                    xaxis_title='Dimension 1',
                    yaxis_title='Dimension 2', 
                    zaxis_title='Dimension 3'
                ),
                width=1000,
                height=800
            )
            
            fig.show()
            print("✓ 3D Plotly visualization displayed")
        except Exception as e:
            print(f"❌ 3D Plotly visualization failed: {e}")
            print("💡 Try installing: pip install nbformat>=4.2.0")
    else:
        print("📊 3D matplotlib visualization not available (use show_plotly=True for interactive 3D)")
    
    return embeddings_3d

# =============================================================================
# Analysis Functions
# =============================================================================

def analyze_embedding_statistics(entity_embeddings, relation_embeddings):
    """Analyze basic statistics of embeddings"""
    print("📈 Embedding Statistics:")
    print(f"Entity embeddings: {entity_embeddings.shape}")
    print(f"  Mean: {entity_embeddings.mean():.4f}")
    print(f"  Std: {entity_embeddings.std():.4f}")
    print(f"  Min: {entity_embeddings.min():.4f}")
    print(f"  Max: {entity_embeddings.max():.4f}")
    
    print(f"\nRelation embeddings: {relation_embeddings.shape}")
    print(f"  Mean: {relation_embeddings.mean():.4f}")
    print(f"  Std: {relation_embeddings.std():.4f}")
    print(f"  Min: {relation_embeddings.min():.4f}")
    print(f"  Max: {relation_embeddings.max():.4f}")

def find_similar_entities(entity_embeddings, id_to_entity, target_entity_id, top_k=10):
    """Find most similar entities to a target entity"""
    target_embedding = entity_embeddings[target_entity_id]
    
    # Compute cosine similarity
    similarities = np.dot(entity_embeddings, target_embedding) / (
        np.linalg.norm(entity_embeddings, axis=1) * np.linalg.norm(target_embedding)
    )
    
    # Get top-k similar entities (excluding self)
    similar_indices = np.argsort(similarities)[::-1][1:top_k+1]
    
    print(f"Most similar entities to '{id_to_entity[target_entity_id]}':")
    for i, idx in enumerate(similar_indices):
        print(f"  {i+1}. {id_to_entity[idx]} (similarity: {similarities[idx]:.4f})")
    
    return similar_indices, similarities[similar_indices]

# =============================================================================
# Relationship-focused Visualization Functions
# =============================================================================

def create_combined_embedding_space(entity_embeddings, relation_embeddings, id_to_entity, id_to_relation, 
                                   entity_colors, entity_labels, sample_entities=2000, sample_relations=None):
    """Create visualization showing both entities and relations in the same space"""
    print("🔗 Creating combined entity-relation embedding space...")
    
    # Sample entities
    if len(entity_embeddings) > sample_entities:
        entity_indices = np.random.choice(len(entity_embeddings), sample_entities, replace=False)
        sampled_entity_embeddings = entity_embeddings[entity_indices]
        sampled_entity_colors = [entity_colors[i] for i in entity_indices]
        sampled_entity_labels = [entity_labels[i] for i in entity_indices]
        sampled_entity_names = [id_to_entity[i] for i in entity_indices]
    else:
        entity_indices = np.arange(len(entity_embeddings))
        sampled_entity_embeddings = entity_embeddings
        sampled_entity_colors = entity_colors
        sampled_entity_labels = entity_labels
        sampled_entity_names = [id_to_entity[i] for i in entity_indices]
    
    # Use all relations or sample if specified
    if sample_relations and len(relation_embeddings) > sample_relations:
        relation_indices = np.random.choice(len(relation_embeddings), sample_relations, replace=False)
        sampled_relation_embeddings = relation_embeddings[relation_indices]
        sampled_relation_names = [id_to_relation[i] for i in relation_indices]
    else:
        relation_indices = np.arange(len(relation_embeddings))
        sampled_relation_embeddings = relation_embeddings
        sampled_relation_names = [id_to_relation[i] for i in relation_indices]
    
    # Combine embeddings
    combined_embeddings = np.vstack([sampled_entity_embeddings, sampled_relation_embeddings])
    
    # Create labels for combined visualization
    combined_labels = sampled_entity_labels + ['Relation'] * len(sampled_relation_embeddings)
    combined_names = sampled_entity_names + [f"REL: {name.split('/')[-1].split('#')[-1]}" for name in sampled_relation_names]
    
    # Create colors (relations get a distinctive color)
    relation_color = '#FF1493'  # Deep pink for relations
    combined_colors = sampled_entity_colors + [relation_color] * len(sampled_relation_embeddings)
    
    # Apply t-SNE
    print("Applying t-SNE to combined space...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(combined_embeddings)-1))
    combined_2d = tsne.fit_transform(combined_embeddings)
    
    # Create visualization
    plt.figure(figsize=(16, 12))
    
    # Plot entities by type
    unique_entity_labels = list(set(sampled_entity_labels))
    for label in unique_entity_labels:
        mask = np.array(combined_labels) == label
        if np.any(mask):
            color = sampled_entity_colors[0] if label in sampled_entity_labels else '#CCCCCC'
            for i, entity_label in enumerate(sampled_entity_labels):
                if entity_label == label:
                    color = sampled_entity_colors[i]
                    break
            plt.scatter(combined_2d[mask, 0], combined_2d[mask, 1], 
                       c=color, label=f"Entity: {label}", alpha=0.6, s=20)
    
    # Plot relations with distinctive markers
    relation_mask = np.array(combined_labels) == 'Relation'
    if np.any(relation_mask):
        plt.scatter(combined_2d[relation_mask, 0], combined_2d[relation_mask, 1], 
                   c=relation_color, label='Relations', alpha=0.8, s=100, marker='^', edgecolors='black')
        
        # Add relation labels
        relation_positions = combined_2d[relation_mask]
        for i, (x, y) in enumerate(relation_positions):
            rel_name = sampled_relation_names[i].split('/')[-1].split('#')[-1][:15]
            plt.annotate(rel_name, (x, y), xytext=(5, 5), textcoords='offset points', 
                        fontsize=8, alpha=0.7, bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.7))
    
    plt.title('Combined Entity-Relation Embedding Space', fontsize=16)
    plt.xlabel('t-SNE Dimension 1', fontsize=12)
    plt.ylabel('t-SNE Dimension 2', fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return combined_2d, combined_labels, combined_names

def create_relation_similarity_heatmap(relation_embeddings, id_to_relation, top_k=20):
    """Create heatmap showing relation similarity"""
    print("📊 Creating relation similarity heatmap...")
    
    # Take top-k most common relations or sample
    n_relations = min(top_k, len(relation_embeddings))
    if len(relation_embeddings) > top_k:
        indices = np.random.choice(len(relation_embeddings), n_relations, replace=False)
        selected_embeddings = relation_embeddings[indices]
        selected_names = [id_to_relation[i] for i in indices]
    else:
        selected_embeddings = relation_embeddings
        selected_names = [id_to_relation[i] for i in range(len(id_to_relation))]
    
    # Compute cosine similarity matrix
    from sklearn.metrics.pairwise import cosine_similarity
    similarity_matrix = cosine_similarity(selected_embeddings)
    
    # Shorten relation names for display
    short_names = [name.split('/')[-1].split('#')[-1][:20] for name in selected_names]
    
    # Create heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(similarity_matrix, 
                xticklabels=short_names, 
                yticklabels=short_names,
                annot=True, 
                fmt='.2f', 
                cmap='RdYlBu_r',
                center=0,
                square=True)
    
    plt.title(f'Relation Similarity Heatmap (Top {n_relations} Relations)', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    return similarity_matrix, short_names

def analyze_entity_relation_patterns(ttl_file="oreganov2.1_metadata_complete.ttl", 
                                   entity_embeddings=None, relation_embeddings=None,
                                   id_to_entity=None, id_to_relation=None):
    """Analyze which entities participate in which relations"""
    print("🔍 Analyzing entity-relation interaction patterns...")
    
    if ttl_file:
        try:
            g = rdflib.Graph()
            g.parse(ttl_file, format="turtle")
            
            # Count entity participation in each relation
            entity_relation_counts = defaultdict(lambda: defaultdict(int))
            relation_popularity = defaultdict(int)
            
            for s, p, o in g:
                if isinstance(o, rdflib.URIRef):  # Only URI objects, not literals
                    head = str(s)
                    relation = str(p)
                    tail = str(o)
                    
                    entity_relation_counts[head][relation] += 1
                    entity_relation_counts[tail][relation] += 1
                    relation_popularity[relation] += 1
            
            # Get top relations
            top_relations = dict(Counter(relation_popularity).most_common(10))
            
            print("📈 Most popular relations:")
            for rel, count in top_relations.items():
                rel_name = rel.split('#')[-1] if '#' in rel else rel.split('/')[-1]
                print(f"  {rel_name}: {count:,} occurrences")
            
            return entity_relation_counts, top_relations
            
        except Exception as e:
            print(f"❌ Could not analyze TTL file: {e}")
            return None, None
    
    print("💡 TTL file not available for relation pattern analysis")
    return None, None

def create_relation_network_graph(relation_embeddings, id_to_relation, similarity_threshold=0.7, max_nodes=30):
    """Create network graph showing relation similarities"""
    print("🕸️ Creating relation similarity network...")
    
    # Sample relations if too many
    n_relations = min(max_nodes, len(relation_embeddings))
    if len(relation_embeddings) > max_nodes:
        indices = np.random.choice(len(relation_embeddings), n_relations, replace=False)
        selected_embeddings = relation_embeddings[indices]
        selected_names = [id_to_relation[i] for i in indices]
    else:
        selected_embeddings = relation_embeddings
        selected_names = [id_to_relation[i] for i in range(len(id_to_relation))]
    
    # Compute similarities
    from sklearn.metrics.pairwise import cosine_similarity
    similarities = cosine_similarity(selected_embeddings)
    
    # Create network graph
    G = nx.Graph()
    
    # Add nodes
    short_names = {}
    for i, name in enumerate(selected_names):
        short_name = name.split('/')[-1].split('#')[-1][:15]
        short_names[i] = short_name
        G.add_node(i, name=short_name, full_name=name)
    
    # Add edges above threshold
    for i in range(len(similarities)):
        for j in range(i+1, len(similarities)):
            if similarities[i][j] > similarity_threshold:
                G.add_edge(i, j, weight=similarities[i][j])
    
    # Create visualization
    plt.figure(figsize=(14, 10))
    
    # Use spring layout for better visualization
    pos = nx.spring_layout(G, k=2, iterations=50)
    
    # Draw edges
    edges = G.edges()
    weights = [G[u][v]['weight'] for u, v in edges]
    nx.draw_networkx_edges(G, pos, alpha=0.5, width=[w*3 for w in weights], edge_color='gray')
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1000, alpha=0.8)
    
    # Draw labels
    labels = {i: short_names[i] for i in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels, font_size=8)
    
    plt.title(f'Relation Similarity Network (threshold > {similarity_threshold})', fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Network contains {G.number_of_nodes()} relations and {G.number_of_edges()} connections")
    
    return G, similarities

def visualize_specific_relation_context(relation_name, ttl_file="oreganov2.1_metadata_complete.ttl",
                                      entity_embeddings=None, id_to_entity=None, sample_size=500, show_plotly=True):
    """Visualize entities that participate in a specific relation with type-based coloring"""
    print(f"🎯 Analyzing entities involved in relation: {relation_name}")
    
    if not ttl_file:
        print("❌ TTL file required for this analysis")
        return None, None
    
    try:
        g = rdflib.Graph()
        g.parse(ttl_file, format="turtle")
        
        # Find entities involved in this relation
        involved_entities = set()
        relation_triples = []
        
        for s, p, o in g:
            pred_str = str(p)
            if relation_name.lower() in pred_str.lower():
                involved_entities.add(str(s))
                if isinstance(o, rdflib.URIRef):
                    involved_entities.add(str(o))
                relation_triples.append((str(s), pred_str, str(o)))
        
        print(f"Found {len(involved_entities)} entities involved in '{relation_name}'")
        print(f"Found {len(relation_triples)} triples with this relation")
        
        if entity_embeddings is not None and id_to_entity is not None:
            # Get entity types for coloring
            print("🎨 Analyzing entity types for coloring...")
            entity_types, type_counts = analyze_entity_types(ttl_file)
            all_entity_colors, all_entity_labels = get_entity_colors_by_type(entity_types, id_to_entity)
            
            # Get embeddings for involved entities
            entity_to_id = {v: k for k, v in id_to_entity.items()}
            involved_indices = []
            involved_names = []
            involved_full_names = []
            involved_colors = []
            involved_types = []
            
            for entity in involved_entities:
                if entity in entity_to_id:
                    entity_id = entity_to_id[entity]
                    involved_indices.append(entity_id)
                    short_name = entity.split('/')[-1].split('#')[-1]
                    involved_names.append(short_name)
                    involved_full_names.append(entity)
                    involved_colors.append(all_entity_colors[entity_id])
                    involved_types.append(all_entity_labels[entity_id])
            
            if len(involved_indices) == 0:
                print("❌ No matching entities found in embeddings")
                return None, None
            
            # Sample if too many (maintaining color/type correspondence)
            if len(involved_indices) > sample_size:
                sample_idx = np.random.choice(len(involved_indices), sample_size, replace=False)
                involved_indices = [involved_indices[i] for i in sample_idx]
                involved_names = [involved_names[i] for i in sample_idx]
                involved_full_names = [involved_full_names[i] for i in sample_idx]
                involved_colors = [involved_colors[i] for i in sample_idx]
                involved_types = [involved_types[i] for i in sample_idx]
            
            # Get embeddings for these entities
            involved_embeddings = entity_embeddings[involved_indices]
            
            # Apply t-SNE
            if len(involved_embeddings) > 1:
                perplexity = min(15, len(involved_embeddings)-1)
                tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
                embeddings_2d = tsne.fit_transform(involved_embeddings)
                
                # Create DataFrame for plotting
                df = pd.DataFrame({
                    'x': embeddings_2d[:, 0],
                    'y': embeddings_2d[:, 1],
                    'entity_name': involved_names,
                    'full_entity': involved_full_names,
                    'entity_type': involved_types,
                    'color': involved_colors,
                    'relation': [relation_name] * len(involved_names)
                })
                
                # Print type distribution
                type_counts_involved = pd.Series(involved_types).value_counts()
                print(f"\n📊 Entity type distribution in '{relation_name}':")
                for entity_type, count in type_counts_involved.items():
                    print(f"  {entity_type}: {count} entities")
                
                if show_plotly:
                    try:
                        # Create interactive Plotly visualization with colors
                        fig = px.scatter(df, 
                                       x='x', y='y',
                                       color='entity_type',
                                       hover_data=['entity_name', 'full_entity'],
                                       title=f'Entities involved in relation: {relation_name} (colored by type)',
                                       labels={'x': 't-SNE Dimension 1', 'y': 't-SNE Dimension 2'},
                                       width=1000, height=700)
                        
                        # Customize markers
                        fig.update_traces(
                            marker=dict(
                                size=10,
                                opacity=0.8,
                                line=dict(width=1, color='DarkSlateGrey')
                            ),
                            hovertemplate='<b>%{customdata[0]}</b><br>' +
                                        'Type: %{customdata[2]}<br>' +
                                        'Full URI: %{customdata[1]}<br>' +
                                        'X: %{x:.2f}<br>' +
                                        'Y: %{y:.2f}<extra></extra>',
                            customdata=df[['entity_name', 'full_entity', 'entity_type']].values
                        )
                        
                        # Add some annotations for key entities (spread across types)
                        if len(df) <= 50:  # Only annotate if not too many points
                            unique_types = df['entity_type'].unique()
                            annotations_per_type = max(1, 10 // len(unique_types))
                            
                            for entity_type in unique_types:
                                type_df = df[df['entity_type'] == entity_type]
                                n_annotations = min(annotations_per_type, len(type_df))
                                
                                for i in range(n_annotations):
                                    row = type_df.iloc[i]
                                    fig.add_annotation(
                                        x=row['x'], y=row['y'],
                                        text=row['entity_name'][:12],
                                        showarrow=True,
                                        arrowhead=2,
                                        arrowsize=1,
                                        arrowwidth=1,
                                        arrowcolor="gray",
                                        ax=20, ay=-20,
                                        font=dict(size=9, color="black"),
                                        bgcolor="rgba(255,255,255,0.9)",
                                        bordercolor="gray",
                                        borderwidth=1
                                    )
                        
                        fig.update_layout(
                            plot_bgcolor='white',
                            xaxis=dict(showgrid=True, gridcolor='lightgray'),
                            yaxis=dict(showgrid=True, gridcolor='lightgray'),
                            legend=dict(
                                orientation="v",
                                yanchor="top",
                                y=1,
                                xanchor="left",
                                x=1.02
                            )
                        )
                        
                        fig.show()
                        print("✓ Interactive Plotly visualization with entity types displayed")
                        
                    except Exception as e:
                        print(f"❌ Plotly visualization failed: {e}")
                        print("📊 Showing matplotlib version instead...")
                        show_plotly = False
                
                # Matplotlib version (fallback or if Plotly disabled)
                if not show_plotly:
                    plt.figure(figsize=(14, 10))
                    
                    # Plot by entity type with different colors
                    unique_types = list(set(involved_types))
                    type_to_color = {}
                    
                    for i, entity_type in enumerate(unique_types):
                        # Get color for this type
                        type_mask = np.array(involved_types) == entity_type
                        if np.any(type_mask):
                            color = involved_colors[np.where(type_mask)[0][0]]
                            type_to_color[entity_type] = color
                            
                            plt.scatter(embeddings_2d[type_mask, 0], embeddings_2d[type_mask, 1], 
                                      alpha=0.7, s=80, c=color, 
                                      label=f"{entity_type} ({np.sum(type_mask)})",
                                      edgecolors='black', linewidth=0.5)
                    
                    # Add labels for a subset of points (spread across types)
                    annotations_added = 0
                    max_annotations = 15
                    annotations_per_type = max(1, max_annotations // len(unique_types))
                    
                    for entity_type in unique_types:
                        type_indices = [i for i, t in enumerate(involved_types) if t == entity_type]
                        n_annotations = min(annotations_per_type, len(type_indices))
                        
                        for i in range(n_annotations):
                            if annotations_added >= max_annotations:
                                break
                            idx = type_indices[i]
                            plt.annotate(involved_names[idx][:12], 
                                       embeddings_2d[idx], 
                                       xytext=(5, 5), 
                                       textcoords='offset points', 
                                       fontsize=8, 
                                       alpha=0.8,
                                       bbox=dict(boxstyle="round,pad=0.3", 
                                               facecolor='white', alpha=0.8))
                            annotations_added += 1
                    
                    plt.title(f'Entities involved in relation: {relation_name} (colored by type)', fontsize=14)
                    plt.xlabel('t-SNE Dimension 1', fontsize=12)
                    plt.ylabel('t-SNE Dimension 2', fontsize=12)
                    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                    plt.grid(True, alpha=0.3)
                    plt.tight_layout()
                    plt.show()
                
                # Print some statistics
                print(f"\n📊 Analysis Summary:")
                print(f"  • Total entities: {len(involved_entities)}")
                print(f"  • Entities with embeddings: {len(involved_indices)}")
                print(f"  • Visualized entities: {len(embeddings_2d)}")
                print(f"  • Entity types found: {len(unique_types)}")
                print(f"  • Sample entities by type:")
                for entity_type in unique_types[:5]:
                    type_entities = [name for name, t in zip(involved_names, involved_types) if t == entity_type]
                    print(f"    - {entity_type}: {', '.join(type_entities[:3])}")
                
                return embeddings_2d, df
            else:
                print("❌ Not enough entities for visualization")
                return None, None
        else:
            print("❌ Entity embeddings and id_to_entity mapping required")
            return None, None
        
    except Exception as e:
        print(f"❌ Error analyzing relation context: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# =============================================================================
# Convenience Functions for Notebook Use
# =============================================================================

def comprehensive_relationship_analysis(sample_entities=2000, sample_relations=50):
    """Complete analysis of both entities and relationships"""
    print("🚀 Starting comprehensive relationship analysis...")
    
    # Load data
    embeddings_data = load_embeddings()
    if embeddings_data[0] is None:
        return
    
    entity_embeddings, relation_embeddings, entity_to_id, relation_to_id, id_to_entity, id_to_relation = embeddings_data
    
    # Get entity types and colors
    entity_types, type_counts = analyze_entity_types()
    entity_colors, entity_labels = get_entity_colors_by_type(entity_types, id_to_entity)
    
    print("\n" + "="*60)
    print("1. 🎨 COMBINED ENTITY-RELATION SPACE")
    print("="*60)
    combined_2d, combined_labels, combined_names = create_combined_embedding_space(
        entity_embeddings, relation_embeddings, id_to_entity, id_to_relation,
        entity_colors, entity_labels, sample_entities=sample_entities, sample_relations=sample_relations
    )
    
    print("\n" + "="*60)
    print("2. 📊 RELATION SIMILARITY HEATMAP")  
    print("="*60)
    similarity_matrix, relation_names = create_relation_similarity_heatmap(
        relation_embeddings, id_to_relation, top_k=min(20, len(relation_embeddings))
    )
    
    print("\n" + "="*60)
    print("3. 🕸️ RELATION NETWORK")
    print("="*60)
    G, similarities = create_relation_network_graph(
        relation_embeddings, id_to_relation, similarity_threshold=0.6, max_nodes=25
    )
    
    print("\n" + "="*60)
    print("4. 🔍 ENTITY-RELATION PATTERNS")
    print("="*60)
    entity_relation_counts, top_relations = analyze_entity_relation_patterns(
        entity_embeddings=entity_embeddings, relation_embeddings=relation_embeddings,
        id_to_entity=id_to_entity, id_to_relation=id_to_relation
    )
    
    return {
        'combined_2d': combined_2d,
        'similarity_matrix': similarity_matrix, 
        'relation_network': G,
        'entity_relation_patterns': entity_relation_counts
    }

def quick_relationship_viz(sample_size=1500):
    """Quick relationship-focused visualization"""
    embeddings_data = load_embeddings()
    if embeddings_data[0] is None:
        return
        
    entity_embeddings, relation_embeddings, entity_to_id, relation_to_id, id_to_entity, id_to_relation = embeddings_data
    entity_types, type_counts = analyze_entity_types()
    entity_colors, entity_labels = get_entity_colors_by_type(entity_types, id_to_entity)
    
    # Show the combined space
    return create_combined_embedding_space(
        entity_embeddings, relation_embeddings, id_to_entity, id_to_relation,
        entity_colors, entity_labels, sample_entities=sample_size
    )
    """Quick visualization with default settings - works without TTL file"""
    # Load data
    embeddings_data = load_embeddings()
    if embeddings_data[0] is None:
        return
    
    entity_embeddings, relation_embeddings, entity_to_id, relation_to_id, id_to_entity, id_to_relation = embeddings_data
    
    # Try to analyze types from TTL, fallback to name inference
    entity_types, type_counts = analyze_entity_types()
    entity_colors, entity_labels = get_entity_colors_by_type(entity_types, id_to_entity)
    
    # Show some sample entities and their inferred types
    print(f"\n📋 Sample entity classifications:")
    sample_ids = list(range(min(10, len(id_to_entity))))
    for i in sample_ids:
        entity_name = id_to_entity[i][:50] + ('...' if len(id_to_entity[i]) > 50 else '')
        print(f"  {entity_labels[i]}: {entity_name}")
    
    # Create visualization
    embeddings_2d, df = create_tsne_visualization(
        entity_embeddings, id_to_entity, entity_colors, entity_labels, 
        sample_size=sample_size, title_suffix=" (Type-based)", show_plotly=show_plotly
    )
    
    return embeddings_2d, df

def compare_visualization_methods(sample_size=2000):
    """Compare different visualization methods side by side"""
    # Load data
    embeddings_data = load_embeddings()
    if embeddings_data[0] is None:
        return
    
    entity_embeddings, relation_embeddings, entity_to_id, relation_to_id, id_to_entity, id_to_relation = embeddings_data
    
    # Get different colorings
    entity_types, _ = analyze_entity_types()
    type_colors, type_labels = get_entity_colors_by_type(entity_types, id_to_entity)
    relation_colors, relation_labels = get_entity_colors_by_relation_involvement(id_to_entity)
    
    # Create side-by-side comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    
    # Sample data
    if len(entity_embeddings) > sample_size:
        indices = np.random.choice(len(entity_embeddings), sample_size, replace=False)
        sampled_embeddings = entity_embeddings[indices]
    else:
        indices = np.arange(len(entity_embeddings))
        sampled_embeddings = entity_embeddings
    
    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(sampled_embeddings)-1))
    embeddings_2d = tsne.fit_transform(sampled_embeddings)
    
    # Plot type-based coloring
    type_colors_sampled = [type_colors[i] for i in indices]
    type_labels_sampled = [type_labels[i] for i in indices]
    unique_type_labels = list(set(type_labels_sampled))
    
    for label in unique_type_labels:
        mask = np.array(type_labels_sampled) == label
        if np.any(mask):
            color = type_colors_sampled[np.where(mask)[0][0]]
            ax1.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                       c=color, label=label, alpha=0.6, s=2)
    
    ax1.set_title('Entity Types')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Plot relation-based coloring
    relation_colors_sampled = [relation_colors[i] for i in indices]
    relation_labels_sampled = [relation_labels[i] for i in indices]
    unique_relation_labels = list(set(relation_labels_sampled))
    
    for label in unique_relation_labels:
        mask = np.array(relation_labels_sampled) == label
        if np.any(mask):
            color = relation_colors_sampled[np.where(mask)[0][0]]
            ax2.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                       c=color, label=label, alpha=0.6, s=2)
    
    ax2.set_title('Dominant Relations')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()
